In [5]:
from huggingface_hub import login
# Đăng nhập Hugging Face
login(token="***REMOVED***")
import os
os.environ["CUDA_DEVICE_ORDER"]      = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]   = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch._dynamo
torch._dynamo.config.disable = True

import logging, gc
import torch
import torch.nn as nn
from typing import Optional, List, Dict, Tuple, Any

from transformers import AutoTokenizer
from transformers.cache_utils import Cache
from transformers import LlamaForCausalLM,  LlamaModel,  LlamaConfig
from transformers import Gemma2ForCausalLM, Gemma2Model, Gemma2Config
from transformers import Qwen2ForCausalLM,  Qwen2Model,  Qwen2Config
from transformers.models.llama.modeling_llama   import LlamaDecoderLayer
from transformers.models.gemma2.modeling_gemma2 import Gemma2DecoderLayer
from transformers.models.qwen2.modeling_qwen2   import Qwen2DecoderLayer

In [6]:
# import pickle
# import numpy as np
# import pandas as pd
# from tabulate import tabulate

# # ────────────────────────────────────────────────────────────
# # Helper functions
# # ────────────────────────────────────────────────────────────

# def load_pkl(path):
#     with open(path, 'rb') as f:
#         return pickle.load(f)

# def compare_vectors(a, b, name_a, name_b):
#     """So sánh chi tiết hai vectors"""
#     # Kiểm tra shape
#     shape_match = a.shape == b.shape
    
#     # Tính cosine similarity
#     a_flat = a.ravel().astype(np.float64)
#     b_flat = b.ravel().astype(np.float64)
#     cosine = np.dot(a_flat, b_flat) / (np.linalg.norm(a_flat) * np.linalg.norm(b_flat))
    
#     # Tính diff
#     diff = a_flat - b_flat
#     max_diff = np.max(np.abs(diff))
#     mean_diff = np.mean(np.abs(diff))
    
#     # Phần trăm giống nhau
#     pct_equal = np.mean(diff == 0) * 100
    
#     # Kết luận
#     is_identical = max_diff < 1e-5
    
#     return {
#         'shape_a': a.shape,
#         'shape_b': b.shape,
#         'shape_match': shape_match,
#         'dtype_a': a.dtype,
#         'dtype_b': b.dtype,
#         'cosine': cosine,
#         'max_diff': max_diff,
#         'mean_diff': mean_diff,
#         'pct_equal': pct_equal,
#         'identical': is_identical
#     }

# def print_comparison_table(results):
#     """In kết quả dạng bảng đẹp"""
#     print("\n" + "="*100)
#     print("📊 COMPARISON RESULTS: Refusal Vector vs Mean Diff Vector")
#     print("="*100)
    
#     df_data = []
#     for model, res in results.items():
#         df_data.append([
#             model,
#             f"{res['shape_a']}",
#             f"{res['dtype_a']}",
#             f"{res['cosine']:.10f}",
#             f"{res['max_diff']:.2e}",
#             f"{res['mean_diff']:.2e}",
#             f"{res['pct_equal']:.2f}%",
#             "✅ IDENTICAL" if res['identical'] else "❌ DIFFERENT"
#         ])
    
#     df = pd.DataFrame(df_data, columns=[
#         'Model', 'Shape', 'Dtype', 'Cosine Sim', 'Max Diff', 'Mean Diff', 'Equal %', 'Status'
#     ])
    
#     print(tabulate(df, headers='keys', tablefmt='grid', showindex=False, stralign='center'))
    
#     # Summary
#     print("\n" + "─"*100)
#     identical_count = sum(1 for res in results.values() if res['identical'])
#     print(f"📌 SUMMARY: {identical_count}/{len(results)} pairs are IDENTICAL")
#     print("─"*100)

# def quick_stats(a, name):
#     """Thống kê nhanh của một vector"""
#     a_flat = a.ravel().astype(np.float64)
#     return {
#         'mean': np.mean(a_flat),
#         'std': np.std(a_flat),
#         'min': np.min(a_flat),
#         'max': np.max(a_flat),
#         'norm': np.linalg.norm(a_flat)
#     }

# # ────────────────────────────────────────────────────────────
# # Main execution
# # ────────────────────────────────────────────────────────────

# # Định nghĩa 3 cặp cần so sánh
# pairs = [
#     {
#         'model': 'Llama 3.1',
#         'path_a': '/home/workspace/mad_workspace/llm/AlphaSteer/data/refusal_vectors/RV/llama3.1_RV_refusal.pkl',
#         'path_b': '/home/workspace/mad_workspace/llm/AdaSteer/vectors/llama31-8b-instruct/RD/mean_diff.pkl'
#     },
#     {
#         'model': 'Qwen 2.5',
#         'path_a': '/home/workspace/mad_workspace/llm/AlphaSteer/data/refusal_vectors/RV/qwen2.5_RV_refusal.pkl',
#         'path_b': '/home/workspace/mad_workspace/llm/AdaSteer/vectors/qwen25-7b-instruct/RD/mean_diff.pkl'
#     },
#     {
#         'model': 'Gemma 2',
#         'path_a': '/home/workspace/mad_workspace/llm/AlphaSteer/data/refusal_vectors/RV/gemma2_RV_refusal.pkl',
#         'path_b': '/home/workspace/mad_workspace/llm/AdaSteer/vectors/gemma2-9b-it/RD/mean_diff.pkl'
#     }
# ]

# # Lưu kết quả
# results = {}

# print("\n🔄 Loading and comparing vectors...")

# for pair in pairs:
#     model = pair['model']
#     print(f"  📂 Processing {model}...")
    
#     a = load_pkl(pair['path_a'])
#     b = load_pkl(pair['path_b'])
    
#     # So sánh
#     results[model] = compare_vectors(a, b, 'RV', 'Mean Diff')
    
#     # Thêm stats chi tiết
#     results[model]['stats_a'] = quick_stats(a, 'RV')
#     results[model]['stats_b'] = quick_stats(b, 'Mean Diff')

# # Hiển thị kết quả chính
# print_comparison_table(results)

# # ────────────────────────────────────────────────────────────
# # Hiển thị thống kê chi tiết cho từng model
# # ────────────────────────────────────────────────────────────

# print("\n" + "="*100)
# print("📈 DETAILED STATISTICS PER MODEL")
# print("="*100)

# for model, res in results.items():
#     print(f"\n🔷 {model}")
#     print(f"  Shape: {res['shape_a']} | Dtype: {res['dtype_a']}")
    
#     # So sánh stats
#     stats_a = res['stats_a']
#     stats_b = res['stats_b']
    
#     stat_df = pd.DataFrame({
#         'Metric': ['Mean', 'Std', 'Min', 'Max', 'L2 Norm'],
#         'Refusal Vector': [f"{stats_a['mean']:.6f}", f"{stats_a['std']:.6f}", 
#                           f"{stats_a['min']:.6f}", f"{stats_a['max']:.6f}", f"{stats_a['norm']:.4f}"],
#         'Mean Diff': [f"{stats_b['mean']:.6f}", f"{stats_b['std']:.6f}", 
#                      f"{stats_b['min']:.6f}", f"{stats_b['max']:.6f}", f"{stats_b['norm']:.4f}"],
#         'Equal?': ['✅' if stats_a[k] == stats_b[k] else '❌' for k in ['mean', 'std', 'min', 'max', 'norm']]
#     })
    
#     print(tabulate(stat_df, headers='keys', tablefmt='simple', showindex=False, stralign='center'))
    
#     # Cosine similarity highlight
#     cos_color = "🟢" if res['cosine'] > 0.9999 else "🟡" if res['cosine'] > 0.99 else "🔴"
#     print(f"  {cos_color} Cosine Similarity: {res['cosine']:.12f}")
#     print(f"  {'✅ Vectors are IDENTICAL' if res['identical'] else '❌ Vectors are DIFFERENT'}")

# print("\n" + "="*100)
# print("🎯 FINAL CONCLUSION")
# print("="*100)

# if all(res['identical'] for res in results.values()):
#     print("✅ TẤT CẢ 3 MODEL đều có Refusal Vector và Mean Diff Vector GIỐNG HỆT nhau!")
#     print("   → mean_diff.pkl chính là bản sao chính xác của refusal vector")
#     print("   → Có thể xóa bớt 1 file để tiết kiệm dung lượng")
# elif any(res['identical'] for res in results.values()):
#     identical_models = [m for m, r in results.items() if r['identical']]
#     diff_models = [m for m, r in results.items() if not r['identical']]
#     print(f"✅ Các model IDENTICAL: {', '.join(identical_models)}")
#     print(f"❌ Các model DIFFERENT: {', '.join(diff_models)}")
# else:
#     print("❌ TẤT CẢ các model đều KHÁC NHAU!")
#     print("   → Mỗi file là một vector độc lập")

# print("="*100 + "\n")

In [7]:
# """
# diagnose_rfm_vs_dim.py
# ======================
# Experiment chẩn đoán toàn diện: RFM-AGOP direction vs DIM direction.
# Chạy 1 shot, không cần args.

# MỤC TIÊU:
#   1. Cosine similarity giữa c (RFM) và r (DIM) theo từng layer, từng model
#   2. L2 norm của activations → lý giải bandwidth mismatch
#   3. Linear separability: Fisher LDA score harmful vs benign
#   4. AUC của RFM direction vs DIM direction như linear probe
#   5. Bandwidth sensitivity analysis cho RFM

# KẾT QUẢ:
#   - In bảng tóm tắt + interpretation ra console
#   - Lưu layer_results.csv, summary_results.csv, diagnostic_report.png
#     vào OUTPUT_DIR
# """

# import os
# import gc
# import pickle
# import logging
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import torch
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# from pathlib import Path

# # ── Logging ───────────────────────────────────────────────────────────────────
# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s [%(levelname)s] %(message)s",
#     datefmt="%H:%M:%S",
# )
# logger = logging.getLogger(__name__)

# # ═════════════════════════════════════════════════════════════════════════════
# # CONFIG — chỉnh paths ở đây
# # ═════════════════════════════════════════════════════════════════════════════

# PROJECT_DIR = Path("/home/workspace/mad_workspace/llm/AGOPNullSpace")

# EMBEDDING_DIRS = {
#     "llama3.1": PROJECT_DIR / "data/embeddings/llama3.1",
#     "qwen2.5":  PROJECT_DIR / "data/embeddings/qwen2.5",
#     "gemma2":   PROJECT_DIR / "data/embeddings/gemma2",
# }

# DIM_PKL_PATHS = {
#     "llama3.1": PROJECT_DIR / "data/refusal_vectors/RV/llama3.1_RV_refusal.pkl",
#     "qwen2.5":  PROJECT_DIR / "data/refusal_vectors/RV/qwen2.5_RV_refusal.pkl",
#     "gemma2":   PROJECT_DIR / "data/refusal_vectors/RV/gemma2_RV_refusal.pkl",
# }

# RFM_PKL_PATHS = {
#     "llama3.1": PROJECT_DIR / "data/refusal_vectors/RFM/llama3.1_RFM_refusal.pkl",
#     "qwen2.5":  PROJECT_DIR / "data/refusal_vectors/RFM/qwen2.5_RFM_refusal.pkl",
#     "gemma2":   PROJECT_DIR / "data/refusal_vectors/RFM/gemma2_RFM_refusal.pkl",
# }

# STEERING_LAYERS = {
#     "llama3.1": [8, 9, 10, 11, 12, 13, 14, 16, 18, 19],
#     "qwen2.5":  [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19],
#     "gemma2":   [6, 8, 10, 11, 12, 13, 14, 15, 16, 18, 22],
# }

# OUTPUT_DIR = Path("./diagnosis_results")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# SAMPLE_N = 500   # samples dùng cho AUC / Fisher / bandwidth estimation

# # ═════════════════════════════════════════════════════════════════════════════
# # HELPERS
# # ═════════════════════════════════════════════════════════════════════════════

# def load_pkl(path):
#     with open(path, "rb") as f:
#         return pickle.load(f)


# def load_embeddings(embed_dir: Path):
#     """Load harmful và benign embeddings, trả về (H_pos, H_neg) float32 CPU."""
#     def _load(fname):
#         p = embed_dir / fname
#         return torch.load(p, map_location="cpu").float() if p.exists() else None

#     H_harmful    = _load("embeds_harmful_train_1000.pt")
#     H_jailbreak  = _load("embeds_jailbreak_train.pt")
#     H_benign     = _load("embeds_benign_train.pt")
#     H_coco_orig  = _load("embeds_coconot_original.pt")
#     H_coco_pref  = _load("embeds_coconot_pref.pt")

#     # Harmful side
#     if H_harmful is None:
#         raise FileNotFoundError(f"embeds_harmful_train_1000.pt not in {embed_dir}")
#     if H_jailbreak is not None:
#         idx = torch.randperm(H_jailbreak.size(0))[:1000]
#         H_pos = torch.cat([H_harmful, H_jailbreak[idx]], dim=0)
#     else:
#         H_pos = H_harmful

#     # Benign side
#     parts = []
#     if H_benign is not None:
#         parts.append(H_benign)
#     if H_coco_orig is not None and H_coco_pref is not None:
#         n_want = 4000 - H_coco_pref.size(0)
#         idx_b = torch.randperm(H_coco_orig.size(0))[:n_want]
#         parts.append(H_coco_orig[idx_b])
#         parts.append(H_coco_pref)
#     elif H_coco_orig is not None:
#         parts.append(H_coco_orig)
#     if not parts:
#         raise FileNotFoundError(f"No benign embeddings in {embed_dir}")
#     H_neg = torch.cat(parts, dim=0)

#     return H_pos, H_neg


# def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
#     na, nb = np.linalg.norm(a), np.linalg.norm(b)
#     return float(np.dot(a, b) / (na * nb)) if na > 1e-10 and nb > 1e-10 else 0.0


# def fisher_score(h_pos: np.ndarray, h_neg: np.ndarray,
#                  direction: np.ndarray) -> float:
#     """Fisher LDA score along direction. Higher = better separation."""
#     p_pos = h_pos @ direction
#     p_neg = h_neg @ direction
#     mu_diff = (p_pos.mean() - p_neg.mean()) ** 2
#     var_sum  = p_pos.var() + p_neg.var() + 1e-10
#     return float(mu_diff / var_sum)


# def auc_from_direction(h_pos: np.ndarray, h_neg: np.ndarray,
#                        direction: np.ndarray) -> float:
#     from sklearn.metrics import roc_auc_score
#     scores = np.concatenate([h_pos @ direction, h_neg @ direction])
#     labels = np.concatenate([np.ones(len(h_pos)), np.zeros(len(h_neg))])
#     try:
#         auc = roc_auc_score(labels, scores)
#         return max(auc, 1.0 - auc)
#     except Exception:
#         return 0.5


# def median_bandwidth(X: np.ndarray, n_sample: int = 400) -> float:
#     """Median heuristic: L = median(‖xᵢ-xⱼ‖) / √2"""
#     idx   = np.random.choice(len(X), min(n_sample, len(X)), replace=False)
#     X_sub = X[idx]
#     dists = np.sqrt(((X_sub[:, None, :] - X_sub[None, :, :]) ** 2).sum(-1))
#     upper = dists[np.triu_indices(len(X_sub), k=1)]
#     return float(np.median(upper) / (2 ** 0.5)) if len(upper) else 1.0


# # ═════════════════════════════════════════════════════════════════════════════
# # PER-MODEL DIAGNOSIS
# # ═════════════════════════════════════════════════════════════════════════════

# def diagnose_model(model_name: str) -> dict:
#     logger.info("=" * 60)
#     logger.info("MODEL: %s", model_name)
#     logger.info("=" * 60)

#     r_dim = load_pkl(DIM_PKL_PATHS[model_name]).astype(np.float32)
#     c_rfm = load_pkl(RFM_PKL_PATHS[model_name]).astype(np.float32)
#     logger.info("DIM %s | RFM %s", r_dim.shape, c_rfm.shape)

#     H_pos_full, H_neg_full = load_embeddings(EMBEDDING_DIRS[model_name])
#     logger.info("H_pos %s | H_neg %s",
#                 tuple(H_pos_full.shape), tuple(H_neg_full.shape))

#     results   = {"model": model_name, "layers": []}
#     cos_sims  = []; fisher_dims = []; fisher_rfms = []
#     auc_dims  = []; auc_rfms   = []
#     med_bws   = []; act_norms  = []

#     for layer in STEERING_LAYERS[model_name]:
#         h_pos_l = H_pos_full[:, layer, :].numpy()
#         h_neg_l = H_neg_full[:, layer, :].numpy()

#         n = min(len(h_pos_l), len(h_neg_l), SAMPLE_N)
#         h_pos = h_pos_l[np.random.choice(len(h_pos_l), n, replace=False)]
#         h_neg = h_neg_l[np.random.choice(len(h_neg_l), n, replace=False)]

#         r = r_dim[layer]; c = c_rfm[layer]
#         if np.linalg.norm(r) < 1e-10 or np.linalg.norm(c) < 1e-10:
#             logger.warning("  Layer %d: zero vector, skip", layer)
#             continue

#         r_n = r / np.linalg.norm(r)
#         c_n = c / np.linalg.norm(c)

#         cos    = cosine_sim(r_n, c_n)
#         fs_dim = fisher_score(h_pos, h_neg, r_n)
#         fs_rfm = fisher_score(h_pos, h_neg, c_n)
#         au_dim = auc_from_direction(h_pos, h_neg, r_n)
#         au_rfm = auc_from_direction(h_pos, h_neg, c_n)

#         X_all  = np.concatenate([h_pos, h_neg])
#         med_bw = median_bandwidth(X_all)
#         mean_n = float(np.linalg.norm(X_all, axis=-1).mean())

#         cos_sims.append(cos); fisher_dims.append(fs_dim); fisher_rfms.append(fs_rfm)
#         auc_dims.append(au_dim); auc_rfms.append(au_rfm)
#         med_bws.append(med_bw); act_norms.append(mean_n)

#         results["layers"].append({
#             "layer": layer, "cosine_dim_rfm": round(cos,4),
#             "fisher_dim": round(fs_dim,4), "fisher_rfm": round(fs_rfm,4),
#             "auc_dim": round(au_dim,4), "auc_rfm": round(au_rfm,4),
#             "median_bw": round(med_bw,2), "mean_norm": round(mean_n,2),
#             "suggested_bws": [round(med_bw*s,1) for s in [0.5,1,2,5,10]],
#         })

#         logger.info(
#             "  L%2d | cos=%+.3f | AUC dim=%.3f rfm=%.3f | "
#             "Fisher dim=%.3f rfm=%.3f | ‖h‖=%.1f median_bw=%.1f",
#             layer, cos, au_dim, au_rfm, fs_dim, fs_rfm, mean_n, med_bw,
#         )

#     if cos_sims:
#         mid = len(med_bws) // 2
#         results["summary"] = {
#             "mean_cosine":     round(float(np.mean(cos_sims)), 4),
#             "min_cosine":      round(float(np.min(cos_sims)), 4),
#             "max_cosine":      round(float(np.max(cos_sims)), 4),
#             "mean_auc_dim":    round(float(np.mean(auc_dims)), 4),
#             "mean_auc_rfm":    round(float(np.mean(auc_rfms)), 4),
#             "mean_fisher_dim": round(float(np.mean(fisher_dims)), 4),
#             "mean_fisher_rfm": round(float(np.mean(fisher_rfms)), 4),
#             "mean_act_norm":   round(float(np.mean(act_norms)), 2),
#             "mean_median_bw":  round(float(np.mean(med_bws)), 2),
#             "suggested_bws":   results["layers"][mid]["suggested_bws"],
#         }
#         s = results["summary"]
#         logger.info(
#             "  SUMMARY | cos=%.3f | AUC dim=%.3f rfm=%.3f | "
#             "‖h‖=%.1f median_bw=%.1f | suggested_bws=%s",
#             s["mean_cosine"], s["mean_auc_dim"], s["mean_auc_rfm"],
#             s["mean_act_norm"], s["mean_median_bw"], s["suggested_bws"],
#         )

#     del H_pos_full, H_neg_full; gc.collect()
#     return results


# # ═════════════════════════════════════════════════════════════════════════════
# # VISUALIZATION
# # ═════════════════════════════════════════════════════════════════════════════

# def plot_all(all_results: dict):
#     models = [m for m in all_results if all_results[m].get("layers")]
#     if not models:
#         return

#     n = len(models)
#     fig = plt.figure(figsize=(20, 5 * n))
#     fig.patch.set_facecolor("#0f0f0f")
#     gs  = gridspec.GridSpec(n, 4, figure=fig, hspace=0.65, wspace=0.4)

#     CBKG = "#1a1a1a"; CGRAY = "#444444"
#     C = {"dim":"#4fc3f7","rfm":"#ff8a65","cos":"#a5d6a7","norm":"#ce93d8","bw":"#ffcc80"}

#     for i, model in enumerate(models):
#         res       = all_results[model]
#         layers    = [r["layer"]          for r in res["layers"]]
#         cos_sims  = [r["cosine_dim_rfm"] for r in res["layers"]]
#         auc_dims  = [r["auc_dim"]        for r in res["layers"]]
#         auc_rfms  = [r["auc_rfm"]        for r in res["layers"]]
#         med_bws   = [r["median_bw"]      for r in res["layers"]]
#         act_norms = [r["mean_norm"]       for r in res["layers"]]

#         def style(ax, title):
#             ax.set_facecolor(CBKG)
#             ax.set_title(f"{model}\n{title}", color="white", fontsize=9, pad=4)
#             ax.tick_params(colors="white", labelsize=7)
#             ax.set_xlabel("Layer", color="white", fontsize=8)
#             for sp in ax.spines.values(): sp.set_color(CGRAY)

#         # (a) Cosine similarity
#         ax1 = fig.add_subplot(gs[i, 0])
#         ax1.bar(layers, cos_sims, color=C["cos"], alpha=0.85, width=0.7)
#         ax1.axhline(0,   color="white", lw=0.5, alpha=0.3)
#         ax1.axhline(0.7, color="#ffeb3b", lw=1, ls="--", alpha=0.7, label="0.7")
#         ax1.axhline(-0.7,color="#ffeb3b", lw=1, ls="--", alpha=0.3)
#         ax1.set_ylim(-1.05, 1.05)
#         ax1.legend(fontsize=6, labelcolor="white", facecolor="#2a2a2a", edgecolor="none")
#         style(ax1, "Cosine(DIM, RFM)")

#         # (b) AUC comparison
#         ax2 = fig.add_subplot(gs[i, 1])
#         x = np.arange(len(layers)); w = 0.35
#         ax2.bar(x-w/2, auc_dims, w, label="DIM", color=C["dim"], alpha=0.85)
#         ax2.bar(x+w/2, auc_rfms, w, label="RFM", color=C["rfm"], alpha=0.85)
#         ax2.axhline(0.5, color="white", lw=0.5, alpha=0.3, ls="--")
#         ax2.set_ylim(0.4, 1.02)
#         ax2.set_xticks(x); ax2.set_xticklabels(layers, rotation=45, fontsize=6)
#         ax2.legend(fontsize=7, labelcolor="white", facecolor="#2a2a2a", edgecolor="none")
#         style(ax2, "AUC (linear probe)")

#         # (c) Activation norm
#         ax3 = fig.add_subplot(gs[i, 2])
#         ax3.plot(layers, act_norms, color=C["norm"], marker="o", ms=4, lw=1.5)
#         style(ax3, "Activation L2 Norm")

#         # (d) Median BW heuristic vs tested range
#         ax4 = fig.add_subplot(gs[i, 3])
#         ax4.semilogy(layers, med_bws, color=C["bw"], marker="s", ms=4, lw=1.5,
#                      label="median bw heuristic")
#         for bw, lab in [(1.0,"bw=1"), (10.0,"bw=10"), (100.0,"bw=100")]:
#             ax4.axhline(bw, color="gray", lw=0.8, ls=":", alpha=0.6)
#             ax4.text(layers[-1], bw*1.1, lab, color="gray", fontsize=6, ha="right")
#         ax4.legend(fontsize=7, labelcolor="white", facecolor="#2a2a2a", edgecolor="none")
#         style(ax4, "Median BW Heuristic vs Tested BWs")

#     fig.suptitle("RFM-AGOP vs DIM — Diagnostic Report",
#                  color="white", fontsize=13, y=1.005, fontweight="bold")

#     out = OUTPUT_DIR / "diagnostic_report.png"
#     fig.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
#     plt.close(fig)
#     logger.info("Figure → %s", out)


# # ═════════════════════════════════════════════════════════════════════════════
# # CONSOLE REPORT
# # ═════════════════════════════════════════════════════════════════════════════

# def print_report(all_results: dict):
#     SEP = "─" * 110
#     HDR = (
#         f"{'Model':<12} │ {'cos':>7} │ {'AUC-DIM':>8} │ {'AUC-RFM':>8} │ "
#         f"{'Fish-DIM':>9} │ {'Fish-RFM':>9} │ {'‖h‖':>8} │ "
#         f"{'med_bw':>8} │ suggested BWs"
#     )

#     print(f"\n{SEP}")
#     print("  DIAGNOSTIC SUMMARY — per model (averaged over steering layers)")
#     print(SEP)
#     print(HDR)
#     print(SEP)

#     interps = []
#     for model, res in all_results.items():
#         s = res.get("summary")
#         if not s:
#             print(f"{model:<12} │ (skipped — missing files)")
#             continue
#         print(
#             f"{model:<12} │ {s['mean_cosine']:>7.4f} │ "
#             f"{s['mean_auc_dim']:>8.4f} │ {s['mean_auc_rfm']:>8.4f} │ "
#             f"{s['mean_fisher_dim']:>9.4f} │ {s['mean_fisher_rfm']:>9.4f} │ "
#             f"{s['mean_act_norm']:>8.1f} │ "
#             f"{s['mean_median_bw']:>8.1f} │ {s['suggested_bws']}"
#         )

#         # Interpretation
#         cos = s["mean_cosine"]
#         if cos > 0.85:
#             ci = "✅ nearly identical direction"
#         elif cos > 0.5:
#             ci = "⚠️  moderately aligned"
#         elif cos > 0.0:
#             ci = "❌ weakly aligned — different information"
#         else:
#             ci = "❌❌ opposite direction — sign flip?"

#         auc_d = s["mean_auc_rfm"] - s["mean_auc_dim"]
#         if auc_d > 0.02:
#             ai = f"✅ RFM > DIM  ({auc_d:+.3f})"
#         elif auc_d > -0.02:
#             ai = f"≈  RFM ≈ DIM ({auc_d:+.3f})"
#         else:
#             ai = f"❌ RFM < DIM  ({auc_d:+.3f})"

#         med = s["mean_median_bw"]
#         if med > 100:
#             bi = (
#                 f"🔴 norm~{s['mean_act_norm']:.0f}, "
#                 f"tested BWs [1,10,100] TOO SMALL → kernel likely saturated"
#             )
#         elif med > 10:
#             bi = f"🟡 norm~{s['mean_act_norm']:.0f}, bw=100 borderline"
#         else:
#             bi = f"🟢 norm~{s['mean_act_norm']:.0f}, tested BWs OK"

#         interps.append((model, ci, ai, bi))

#     print(SEP)
#     print("\n  PER-MODEL INTERPRETATION")
#     print(SEP)
#     for model, ci, ai, bi in interps:
#         print(f"\n  [{model}]")
#         print(f"    Direction alignment : {ci}")
#         print(f"    Separability (AUC)  : {ai}")
#         print(f"    Bandwidth situation : {bi}")

#     # Cross-model conclusion
#     done = [m for m, res in all_results.items() if res.get("summary")]
#     if len(done) >= 2:
#         print(f"\n{SEP}")
#         print("  CROSS-MODEL DIAGNOSIS")
#         print(SEP)

#         bw_vals   = {m: all_results[m]["summary"]["mean_median_bw"]  for m in done}
#         cos_vals  = {m: all_results[m]["summary"]["mean_cosine"]      for m in done}
#         auc_diffs = {
#             m: all_results[m]["summary"]["mean_auc_rfm"] -
#                all_results[m]["summary"]["mean_auc_dim"]
#             for m in done
#         }

#         bw_ratio = max(bw_vals.values()) / (min(bw_vals.values()) + 1e-6)
#         print(f"\n  Bandwidth spread: {min(bw_vals.values()):.1f} – "
#               f"{max(bw_vals.values()):.1f}  (ratio = {bw_ratio:.1f}x)")

#         if bw_ratio > 5:
#             print(
#                 "\n  🔴 BANDWIDTH MISMATCH (likely root cause for Qwen/Gemma failures)\n"
#                 "     Activation scale varies >5x across models.\n"
#                 "     Tested BWs [1, 10, 100] work for Llama but are too small\n"
#                 "     for models with larger activation norms.\n"
#                 "\n  RECOMMENDED FIX in rfm_refusal_vector.py:\n"
#                 "     Replace:  for bw in [1.0, 10.0, 100.0]\n"
#                 "     With:\n"
#                 "       X_all = torch.cat([h_pos, h_neg]).numpy()\n"
#                 "       bw_base = median_bandwidth(X_all)  # per layer\n"
#                 "       for bw in [bw_base*0.5, bw_base*1.0, bw_base*2.0, bw_base*5.0]\n"
#             )
#         else:
#             print("\n  🟢 Bandwidth spread OK — BW mismatch is NOT the primary issue.")

#         low_cos = [m for m, c in cos_vals.items() if c < 0.5]
#         if low_cos:
#             print(
#                 f"\n  ⚠️  LOW COSINE for: {low_cos}\n"
#                 "     RFM-AGOP is learning a DIFFERENT direction than DIM.\n"
#                 "     Possible causes:\n"
#                 "       (a) Bandwidth too small → kernel saturated → AGOP noisy\n"
#                 "       (b) Data distribution (2000 harmful vs ~14k benign downsampled)\n"
#                 "           causes AGOP to emphasize different axis than mean-diff\n"
#                 "       (c) Model-specific geometry — harmful/benign less linearly\n"
#                 "           separable for this model → AGOP finds non-refusal axis\n"
#             )

#         worse = [m for m, d in auc_diffs.items() if d < -0.02]
#         if worse:
#             print(
#                 f"\n  ❌ RFM UNDERPERFORMS DIM for: {worse}\n"
#                 "     Weaker refusal direction → steering matrix ∆* suboptimal\n"
#                 "     → larger λ needed to compensate → per-dataset λ tuning required.\n"
#                 "     This is the root cause of the λ instability you observed.\n"
#             )

#     print(f"\n{SEP}\n")


# # ═════════════════════════════════════════════════════════════════════════════
# # CSV SAVE
# # ═════════════════════════════════════════════════════════════════════════════

# def save_csv(all_results: dict):
#     import csv

#     # Layer-level CSV
#     p1 = OUTPUT_DIR / "layer_results.csv"
#     cols1 = ["model","layer","cosine_dim_rfm","fisher_dim","fisher_rfm",
#              "auc_dim","auc_rfm","median_bw","mean_norm"]
#     with open(p1, "w", newline="") as f:
#         w = csv.DictWriter(f, fieldnames=cols1)
#         w.writeheader()
#         for model, res in all_results.items():
#             for lr in res.get("layers", []):
#                 w.writerow({k: lr[k] for k in cols1 if k != "model"} | {"model": model})
#     logger.info("CSV → %s", p1)

#     # Summary CSV
#     p2 = OUTPUT_DIR / "summary_results.csv"
#     cols2 = ["model","mean_cosine","min_cosine","max_cosine",
#              "mean_auc_dim","mean_auc_rfm","mean_fisher_dim","mean_fisher_rfm",
#              "mean_act_norm","mean_median_bw","suggested_bws"]
#     with open(p2, "w", newline="") as f:
#         w = csv.DictWriter(f, fieldnames=cols2)
#         w.writeheader()
#         for model, res in all_results.items():
#             s = res.get("summary")
#             if s:
#                 w.writerow({"model": model, **{k: s[k] for k in cols2[1:]},
#                             "suggested_bws": str(s["suggested_bws"])})
#     logger.info("Summary CSV → %s", p2)

In [8]:
# np.random.seed(42)
# torch.manual_seed(42)

# logger.info("Output dir : %s", OUTPUT_DIR.resolve())
# logger.info("sklearn required for AUC — ensure it is installed.")

# all_results = {}
# for model_name in ["llama3.1", "qwen2.5", "gemma2"]:
#     missing = []
#     if not EMBEDDING_DIRS[model_name].exists():
#         missing.append(f"embed dir  : {EMBEDDING_DIRS[model_name]}")
#     if not DIM_PKL_PATHS[model_name].exists():
#         missing.append(f"DIM pkl    : {DIM_PKL_PATHS[model_name]}")
#     if not RFM_PKL_PATHS[model_name].exists():
#         missing.append(f"RFM pkl    : {RFM_PKL_PATHS[model_name]}")

#     if missing:
#         logger.warning("Skipping %s — missing:\n    %s",
#                        model_name, "\n    ".join(missing))
#         all_results[model_name] = {}
#         continue

#     try:
#         all_results[model_name] = diagnose_model(model_name)
#     except Exception as e:
#         logger.error("Error on %s: %s", model_name, e, exc_info=True)
#         all_results[model_name] = {}

# plot_all(all_results)
# print_report(all_results)

# logger.info("Done. All results in: %s", OUTPUT_DIR.resolve())

# sc_rfm_experiment

In [11]:
!pwd
/home/workspace/mad_workspace/llm/

/home/workspace/mad_workspace/llm_workspace/AGOPNullSpace


In [1]:
"""
sc_rfm_experiment.py
====================
Full implementation + experiment cho Steering-Constrained RFM (SC-RFM).

Chạy 1 shot, không cần args.

THUẬT TOÁN SC-RFM:
  1. Chạy RFM-AGOP bình thường → AGOP matrix M_T ∈ R^{d×d}
  2. Eigendecomposition M_T → top-K eigenvectors {u_k}, eigenvalues {λ_k}
  3. Tính alignment với DIM direction r: align_k = u_k · r
  4. Weighted combination chỉ giữ cùng chiều với r:
       c* = normalize( Σ_k λ_k · relu(u_k · r) · u_k )
  5. Adaptive bandwidth: dùng median heuristic thay vì fixed [1, 10, 100]

EXPERIMENT:
  Với mỗi model, mỗi layer:
    - Tính 4 directions: DIM, RFM (top-1), SC-RFM, SC-RFM-adaptive-bw
    - So sánh: cosine với DIM, AUC, Fisher score
    - Save pkl SC-RFM để dùng trong AlphaSteer pipeline

OUTPUT:
  sc_rfm_results/
    ├── sc_rfm_experiment.csv          ← kết quả chi tiết
    ├── sc_rfm_summary.csv             ← tóm tắt per model
    ├── sc_rfm_report.png              ← visualization
    ├── llama3.1_SCRFM_refusal.pkl     ← steering vector
    ├── qwen2.5_SCRFM_refusal.pkl
    └── gemma2_SCRFM_refusal.pkl
"""

import os
import gc
import pickle
import logging
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from copy import deepcopy
from pathlib import Path

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# ═════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═════════════════════════════════════════════════════════════════════════════

os.environ["CUDA_DEVICE_ORDER"]    = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

PROJECT_DIR = Path("/home/workspace/mad_workspace/llm_workspace/AGOPNullSpace")

EMBEDDING_DIRS = {
    "llama3.1": PROJECT_DIR / "data/embeddings/llama3.1",
    "qwen2.5":  PROJECT_DIR / "data/embeddings/qwen2.5",
    "gemma2":   PROJECT_DIR / "data/embeddings/gemma2",
}

DIM_PKL_PATHS = {
    "llama3.1": PROJECT_DIR / "data/refusal_vectors/RV/llama3.1_RV_refusal.pkl",
    "qwen2.5":  PROJECT_DIR / "data/refusal_vectors/RV/qwen2.5_RV_refusal.pkl",
    "gemma2":   PROJECT_DIR / "data/refusal_vectors/RV/gemma2_RV_refusal.pkl",
}

RFM_PKL_PATHS = {
    "llama3.1": PROJECT_DIR / "data/refusal_vectors/RFM/llama3.1_RFM_refusal.pkl",
    "qwen2.5":  PROJECT_DIR / "data/refusal_vectors/RFM/qwen2.5_RFM_refusal.pkl",
    "gemma2":   PROJECT_DIR / "data/refusal_vectors/RFM/gemma2_RFM_refusal.pkl",
}

STEERING_LAYERS = {
    "llama3.1": [8, 9, 10, 11, 12, 13, 14, 16, 18, 19],
    "qwen2.5":  [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19],
    "gemma2":   [6, 8, 10, 11, 12, 13, 14, 15, 16, 18, 22],
}

OUTPUT_DIR = Path("./sc_rfm_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
SEED     = 42
SAMPLE_N = 500    # samples per class cho AUC/Fisher
TOP_K    = 20     # số eigenvectors dùng trong SC-RFM
RFM_ITERS = 3


# ═════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ═════════════════════════════════════════════════════════════════════════════

def load_pkl(path) -> np.ndarray:
    with open(path, "rb") as f:
        return pickle.load(f)


def load_embeddings(embed_dir: Path):
    """
    Load harmful và benign embeddings.
    Returns (H_pos, H_neg): [N, L, D] float32 CPU tensors.
    """
    def _load(fname):
        p = embed_dir / fname
        return torch.load(p, map_location="cpu").float() if p.exists() else None

    H_harmful   = _load("embeds_harmful_train_1000.pt")
    H_jailbreak = _load("embeds_jailbreak_train.pt")
    H_benign    = _load("embeds_benign_train.pt")
    H_coco_orig = _load("embeds_coconot_original.pt")
    H_coco_pref = _load("embeds_coconot_pref.pt")

    if H_harmful is None:
        raise FileNotFoundError(f"embeds_harmful_train_1000.pt not in {embed_dir}")

    # Harmful side
    if H_jailbreak is not None:
        idx = torch.randperm(H_jailbreak.size(0))[:1000]
        H_pos = torch.cat([H_harmful, H_jailbreak[idx]], dim=0)
    else:
        H_pos = H_harmful

    # Benign side
    parts = []
    if H_benign is not None:
        parts.append(H_benign)
    if H_coco_orig is not None and H_coco_pref is not None:
        n_want = 4000 - H_coco_pref.size(0)
        idx_b  = torch.randperm(H_coco_orig.size(0))[:n_want]
        parts.append(H_coco_orig[idx_b])
        parts.append(H_coco_pref)
    elif H_coco_orig is not None:
        parts.append(H_coco_orig)
    if not parts:
        raise FileNotFoundError(f"No benign embeddings in {embed_dir}")
    H_neg = torch.cat(parts, dim=0)

    return H_pos, H_neg


# ═════════════════════════════════════════════════════════════════════════════
# UTILITIES
# ═════════════════════════════════════════════════════════════════════════════

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 1e-10 and nb > 1e-10 else 0.0


def fisher_score(h_pos: np.ndarray, h_neg: np.ndarray,
                 direction: np.ndarray) -> float:
    """Fisher LDA score along direction. Higher = better separation."""
    p_pos = h_pos @ direction
    p_neg = h_neg @ direction
    mu_diff = (p_pos.mean() - p_neg.mean()) ** 2
    var_sum  = p_pos.var() + p_neg.var() + 1e-10
    return float(mu_diff / var_sum)


def auc_score(h_pos: np.ndarray, h_neg: np.ndarray,
              direction: np.ndarray) -> float:
    from sklearn.metrics import roc_auc_score
    scores = np.concatenate([h_pos @ direction, h_neg @ direction])
    labels = np.concatenate([np.ones(len(h_pos)), np.zeros(len(h_neg))])
    try:
        auc = roc_auc_score(labels, scores)
        return max(auc, 1.0 - auc)
    except Exception:
        return 0.5


def median_bandwidth(X: np.ndarray, n_sample: int = 400) -> float:
    """Silverman/median heuristic: L = median(‖xᵢ-xⱼ‖) / √2"""
    idx   = np.random.choice(len(X), min(n_sample, len(X)), replace=False)
    X_sub = X[idx]
    dists = np.sqrt(((X_sub[:, None] - X_sub[None]) ** 2).sum(-1))
    upper = dists[np.triu_indices(len(X_sub), k=1)]
    return float(np.median(upper) / (2 ** 0.5)) if len(upper) else 1.0


def safe_norm(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    return v / n if n > 1e-10 else v


# ═════════════════════════════════════════════════════════════════════════════
# AGOP COMPUTATION (từ rfm_refusal_vector.py v2, với adaptive BW)
# ═════════════════════════════════════════════════════════════════════════════

def standardize_gpu(X: torch.Tensor):
    mean_ = X.mean(dim=0)
    std_  = X.std(dim=0).clamp(min=1e-8)
    return (X - mean_) / std_, mean_, std_


def _logistic_loss(w, X, y, C=1.0):
    logits = X @ w
    loss   = torch.nn.functional.binary_cross_entropy_with_logits(
        logits, y, reduction="mean")
    return loss + (0.5 / C) * (w @ w)


def train_logistic_gpu(X, y, C=1.0, max_iter=500):
    w = torch.zeros(X.shape[1], dtype=torch.float32,
                    device=X.device, requires_grad=True)
    opt = torch.optim.LBFGS(
        [w], lr=1.0, max_iter=max_iter,
        tolerance_grad=1e-6, tolerance_change=1e-6,
        history_size=10, line_search_fn="strong_wolfe",
    )
    def closure():
        opt.zero_grad()
        loss = _logistic_loss(w, X, y, C=C)
        loss.backward()
        return loss
    opt.step(closure)
    return w.detach()


def compute_agop_matrix_rfm(
    H_pos: torch.Tensor,
    H_neg: torch.Tensor,
    rfm_iters: int = 3,
    device: str = "cpu",
    bandwidths=None,            # None → adaptive median heuristic
) -> tuple:
    """
    Compute AGOP matrix (full d×d) từ RFM.
    Returns (agop_matrix [d,d] CPU, top1_direction [d] CPU).

    Key difference từ v2:
      - bandwidths=None → dùng median heuristic tự động
      - Trả về FULL agop matrix để SC-RFM dùng eigendecomposition
    """
    try:
        from xrfm import RFM
        from sklearn.metrics import roc_auc_score
    except ImportError:
        logger.warning("xrfm not installed → linear AGOP fallback")
        return _compute_agop_linear_full(H_pos, H_neg, device)

    dev    = torch.device(device)
    N_pos  = H_pos.shape[0]
    N_neg  = H_neg.shape[0]

    X = torch.cat([H_pos, H_neg], dim=0).float().to(dev)
    y = torch.cat([torch.ones(N_pos, 1, device=dev),
                   torch.zeros(N_neg, 1, device=dev)])

    # Stratified split
    pos_idx = (y.squeeze() == 1).nonzero(as_tuple=True)[0]
    neg_idx = (y.squeeze() == 0).nonzero(as_tuple=True)[0]
    nv_p    = max(1, int(0.2 * len(pos_idx)))
    nv_n    = max(1, int(0.2 * len(neg_idx)))
    val_idx   = torch.cat([pos_idx[:nv_p], neg_idx[:nv_n]])
    train_idx = torch.cat([pos_idx[nv_p:], neg_idx[nv_n:]])
    Xtr, ytr  = X[train_idx], y[train_idx]
    Xvl, yvl  = X[val_idx],   y[val_idx]

    # Adaptive bandwidth
    if bandwidths is None:
        bw_base = median_bandwidth(X.cpu().numpy())
        bandwidths = [bw_base * s for s in [0.5, 1.0, 2.0, 5.0]]
        logger.info("  Adaptive BW: base=%.1f → %s", bw_base,
                    [round(b, 1) for b in bandwidths])
    else:
        bw_base = bandwidths[0]

    best_model, best_auc = None, -1.0
    for bw in bandwidths:
        for reg in [1e-3, 1e-2]:
            try:
                m = RFM(kernel="l2_high_dim", bandwidth=bw, device=device)
                m.fit((Xtr, ytr), (Xvl, yvl),
                      reg=reg, iters=rfm_iters,
                      center_grads=True, early_stop_rfm=True,
                      get_agop_best_model=True, top_k=1)
                preds = m.predict(Xvl).cpu().numpy()
                auc   = roc_auc_score(yvl.cpu().numpy(), preds)
                if auc > best_auc:
                    best_auc   = auc
                    best_model = deepcopy(m)
            except Exception as e:
                logger.debug("  RFM bw=%.1f reg=%.0e failed: %s", bw, reg, e)

    if best_model is None:
        logger.warning("  All RFM fits failed → linear fallback")
        return _compute_agop_linear_full(H_pos, H_neg, device)

    logger.info("  Best RFM AUC=%.4f", best_auc)
    agop = best_model.agop_best_model.cpu()  # [d, d] tensor

    # Top-1 direction (standard RFM output)
    S, U = torch.lobpcg(agop, k=1)
    r_top1 = U[:, 0]
    proj = X.cpu() @ r_top1
    if torch.corrcoef(torch.stack([proj, y.squeeze().cpu()]))[0, 1] < 0:
        r_top1 = -r_top1

    del X, y, Xtr, ytr, Xvl, yvl
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return agop.numpy().astype(np.float32), r_top1.numpy()


def _compute_agop_linear_full(H_pos, H_neg, device):
    """Linear probe fallback: AGOP = outer(w, w), top1 = normalized w."""
    dev = torch.device(device)
    X   = torch.cat([H_pos, H_neg], dim=0).float().to(dev)
    y   = torch.cat([torch.ones(len(H_pos), device=dev),
                     torch.zeros(len(H_neg), device=dev)])
    X_sc, mean_, std_ = standardize_gpu(X)

    best_acc, best_w = -1.0, None
    for C in [0.1, 1.0, 10.0]:
        w_sc = train_logistic_gpu(X_sc, y, C=C)
        acc  = ((X_sc @ w_sc > 0).float() == y).float().mean().item()
        if acc > best_acc:
            best_acc, best_w = acc, w_sc.clone()

    w_orig = (best_w / std_).cpu().numpy().astype(np.float32)
    agop   = np.outer(w_orig, w_orig)
    r      = w_orig / (np.linalg.norm(w_orig) + 1e-10)

    del X, y, X_sc, mean_, std_, best_w
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return agop, r


# ═════════════════════════════════════════════════════════════════════════════
# SC-RFM: STEERING-CONSTRAINED RFM
# ═════════════════════════════════════════════════════════════════════════════

def sc_rfm(
    agop: np.ndarray,          # [d, d] AGOP matrix
    r_dim: np.ndarray,         # [d]   DIM steering direction (normalized)
    top_k: int = 20,
    fallback_threshold: float = 0.05,
) -> dict:
    """
    Steering-Constrained RFM.

    Từ AGOP matrix, tìm direction vừa discriminative (theo AGOP eigenvalue)
    vừa compatible với steering direction r_dim.

    Formula:
        c* = normalize( Σ_{k=1}^{K} λ_k · relu(u_k · r) · u_k )

    Nếu không có component nào align (sum weights < threshold):
        fallback: dùng component có |align| lớn nhất, flip sign nếu cần.

    Returns dict với:
        'c_scrfm':          direction SC-RFM [d]
        'components_used':  số eigenvectors contribute
        'weights':          [K] weights trước normalize
        'alignments':       [K] cosine(u_k, r) cho top-K
        'eigenvalues':      [K] top-K eigenvalues
        'cos_with_dim':     cosine(c*, r)
        'fallback_used':    bool
    """
    r = safe_norm(r_dim)

    # Eigendecomposition (symmetric matrix → use eigh)
    K_actual = min(top_k, agop.shape[0])
    try:
        # scipy eigh faster for large d
        from scipy.linalg import eigh
        eigenvalues, eigenvectors = eigh(
            agop, subset_by_index=[agop.shape[0] - K_actual, agop.shape[0] - 1]
        )
        # Returns ascending → reverse
        eigenvalues  = eigenvalues[::-1].copy()
        eigenvectors = eigenvectors[:, ::-1].copy()
    except Exception:
        eigenvalues, eigenvectors = np.linalg.eigh(agop)
        idx          = np.argsort(eigenvalues)[::-1]
        eigenvalues  = eigenvalues[idx][:K_actual]
        eigenvectors = eigenvectors[:, idx][:, :K_actual]

    # Clip negative eigenvalues (numerical noise)
    eigenvalues = np.maximum(eigenvalues, 0)

    # Alignment with DIM direction
    alignments = eigenvectors.T @ r   # [K]

    # SC-RFM weights: λ_k · relu(u_k · r)
    weights = eigenvalues * np.maximum(alignments, 0)

    fallback_used = False
    if weights.sum() < fallback_threshold:
        # Fallback: dùng best-aligned component (abs), flip sign nếu cần
        k_best = int(np.argmax(np.abs(alignments)))
        c      = eigenvectors[:, k_best] * np.sign(alignments[k_best])
        fallback_used  = True
        components_used = 1
        logger.warning(
            "  SC-RFM fallback: no aligned component "
            "(max align=%.3f < threshold=%.2f), using best-flip k=%d",
            float(np.max(np.abs(alignments))), fallback_threshold, k_best,
        )
    else:
        c = eigenvectors @ weights       # [d]
        components_used = int((weights > 0).sum())

    c = safe_norm(c)
    cos_with_dim = float(np.dot(c, r))

    return {
        "c_scrfm":         c,
        "components_used": components_used,
        "weights":         weights,
        "alignments":      alignments,
        "eigenvalues":     eigenvalues,
        "cos_with_dim":    cos_with_dim,
        "fallback_used":   fallback_used,
    }


# ═════════════════════════════════════════════════════════════════════════════
# PER-LAYER EXPERIMENT
# ═════════════════════════════════════════════════════════════════════════════

def run_layer(
    layer: int,
    H_pos_full: torch.Tensor,
    H_neg_full: torch.Tensor,
    r_dim_all: np.ndarray,
    c_rfm_all: np.ndarray,
    rfm_iters: int,
    device: str,
    top_k: int,
    sample_n: int,
) -> dict:
    """
    Full experiment cho 1 layer.
    Returns dict kết quả.
    """
    # Extract layer activations
    h_pos_l = H_pos_full[:, layer, :].numpy()
    h_neg_l = H_neg_full[:, layer, :].numpy()

    # Sample cho metric computation
    n  = min(len(h_pos_l), len(h_neg_l), sample_n)
    hp = h_pos_l[np.random.choice(len(h_pos_l), n, replace=False)]
    hn = h_neg_l[np.random.choice(len(h_neg_l), n, replace=False)]

    # DIM direction
    r_dim = safe_norm(r_dim_all[layer].astype(np.float32))

    # RFM top-1 direction (existing)
    c_rfm = safe_norm(c_rfm_all[layer].astype(np.float32))

    # ── Compute AGOP với adaptive bandwidth ───────────────────────────────────
    h_pos_t = torch.tensor(h_pos_l).float()
    h_neg_t = torch.tensor(h_neg_l).float()

    # Balance
    n_min = min(len(h_pos_t), len(h_neg_t))
    if len(h_pos_t) > n_min:
        h_pos_t = h_pos_t[torch.randperm(len(h_pos_t))[:n_min]]
    if len(h_neg_t) > n_min:
        h_neg_t = h_neg_t[torch.randperm(len(h_neg_t))[:n_min]]

    logger.info("  Layer %d: computing AGOP (adaptive BW)...", layer)
    agop, c_rfm_adaptive_top1 = compute_agop_matrix_rfm(
        h_pos_t, h_neg_t,
        rfm_iters=rfm_iters,
        device=device,
        bandwidths=None,   # ← adaptive
    )
    # agop: [d, d] numpy float32
    # c_rfm_adaptive_top1: top-1 eigenvector từ AGOP với adaptive BW

    # ── SC-RFM ────────────────────────────────────────────────────────────────
    sc_result = sc_rfm(agop, r_dim, top_k=top_k)
    c_scrfm   = sc_result["c_scrfm"]

    # ── Metrics cho 4 directions ──────────────────────────────────────────────
    directions = {
        "DIM":              r_dim,
        "RFM_top1_fixed":   c_rfm,                  # existing pkl, fixed BW
        "RFM_top1_adaptive": c_rfm_adaptive_top1,   # new, adaptive BW
        "SC_RFM":           c_scrfm,                # new algorithm
    }

    metrics = {}
    for name, d_vec in directions.items():
        metrics[name] = {
            "auc":          auc_score(hp, hn, d_vec),
            "fisher":       fisher_score(hp, hn, d_vec),
            "cos_with_dim": cosine_sim(d_vec, r_dim),
        }

    # Additional SC-RFM info
    sc_info = {
        "components_used": sc_result["components_used"],
        "cos_sc_with_dim": sc_result["cos_with_dim"],
        "fallback_used":   sc_result["fallback_used"],
        "max_alignment":   float(np.max(sc_result["alignments"])),
        "min_alignment":   float(np.min(sc_result["alignments"])),
        "eigenvalue_top1": float(sc_result["eigenvalues"][0]),
        "eigenvalue_top3": float(sc_result["eigenvalues"][:3].sum()),
    }

    # Norm info
    norm_info = {
        "mean_norm_pos": float(np.linalg.norm(h_pos_l, axis=-1).mean()),
        "mean_norm_neg": float(np.linalg.norm(h_neg_l, axis=-1).mean()),
    }

    return {
        "layer":       layer,
        "metrics":     metrics,
        "sc_info":     sc_info,
        "norm_info":   norm_info,
        "c_scrfm":     c_scrfm,
        "c_rfm_adabw": c_rfm_adaptive_top1,
    }


# ═════════════════════════════════════════════════════════════════════════════
# PER-MODEL EXPERIMENT
# ═════════════════════════════════════════════════════════════════════════════

def run_model(model_name: str) -> dict:
    logger.info("=" * 70)
    logger.info("MODEL: %s", model_name)
    logger.info("=" * 70)

    # Load
    r_dim_all = load_pkl(DIM_PKL_PATHS[model_name]).astype(np.float32)
    c_rfm_all = load_pkl(RFM_PKL_PATHS[model_name]).astype(np.float32)
    H_pos_full, H_neg_full = load_embeddings(EMBEDDING_DIRS[model_name])

    L, D = H_pos_full.shape[1], H_pos_full.shape[2]
    logger.info("Layers_total=%d  D=%d | H_pos=%s  H_neg=%s",
                L, D, tuple(H_pos_full.shape), tuple(H_neg_full.shape))

    layers     = STEERING_LAYERS[model_name]
    layer_results = []
    scrfm_vectors = np.zeros((L, D), dtype=np.float32)  # for pkl save

    for layer in layers:
        try:
            res = run_layer(
                layer, H_pos_full, H_neg_full,
                r_dim_all, c_rfm_all,
                rfm_iters=RFM_ITERS,
                device=DEVICE,
                top_k=TOP_K,
                sample_n=SAMPLE_N,
            )
            layer_results.append(res)
            scrfm_vectors[layer] = res["c_scrfm"]

            m = res["metrics"]
            sc = res["sc_info"]
            logger.info(
                "  L%2d | AUC: DIM=%.3f RFM_fix=%.3f RFM_ada=%.3f SC=%.3f | "
                "cos_dim: RFM_fix=%+.3f RFM_ada=%+.3f SC=%+.3f | "
                "comps=%d fallback=%s",
                layer,
                m["DIM"]["auc"], m["RFM_top1_fixed"]["auc"],
                m["RFM_top1_adaptive"]["auc"], m["SC_RFM"]["auc"],
                m["RFM_top1_fixed"]["cos_with_dim"],
                m["RFM_top1_adaptive"]["cos_with_dim"],
                m["SC_RFM"]["cos_with_dim"],
                sc["components_used"], sc["fallback_used"],
            )
        except Exception as e:
            logger.error("  Layer %d failed: %s", layer, e, exc_info=True)

    # Save SC-RFM pkl
    pkl_out = OUTPUT_DIR / f"{model_name}_SCRFM_refusal.pkl"
    with open(pkl_out, "wb") as f:
        pickle.dump(scrfm_vectors, f)
    logger.info("SC-RFM pkl → %s", pkl_out)

    del H_pos_full, H_neg_full; gc.collect()

    return {
        "model":         model_name,
        "layer_results": layer_results,
        "scrfm_vectors": scrfm_vectors,
    }


# ═════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ═════════════════════════════════════════════════════════════════════════════

def plot_results(all_results: dict):
    models = [m for m in all_results if all_results[m].get("layer_results")]
    if not models:
        return

    n  = len(models)
    fig = plt.figure(figsize=(22, 6 * n))
    fig.patch.set_facecolor("#0d0d0d")
    gs  = gridspec.GridSpec(n, 4, figure=fig, hspace=0.65, wspace=0.38)

    DIR_COLORS = {
        "DIM":              "#78909c",
        "RFM_top1_fixed":   "#ef5350",
        "RFM_top1_adaptive":"#ffa726",
        "SC_RFM":           "#66bb6a",
    }
    DIR_LABELS = {
        "DIM":              "DIM",
        "RFM_top1_fixed":   "RFM (fixed BW)",
        "RFM_top1_adaptive":"RFM (adaptive BW)",
        "SC_RFM":           "SC-RFM (ours)",
    }

    def style(ax, title):
        ax.set_facecolor("#1a1a1a")
        ax.set_title(title, color="white", fontsize=8.5, pad=5)
        ax.tick_params(colors="white", labelsize=7)
        ax.set_xlabel("Layer", color="white", fontsize=8)
        for sp in ax.spines.values():
            sp.set_color("#444")

    for i, model in enumerate(models):
        res    = all_results[model]["layer_results"]
        layers = [r["layer"] for r in res]
        x      = np.arange(len(layers))
        w      = 0.2

        # (a) AUC comparison — 4 bars per layer
        ax1 = fig.add_subplot(gs[i, 0])
        for j, (dname, col) in enumerate(DIR_COLORS.items()):
            aucs = [r["metrics"][dname]["auc"] for r in res]
            ax1.bar(x + (j-1.5)*w, aucs, w, label=DIR_LABELS[dname],
                    color=col, alpha=0.85)
        ax1.axhline(0.5, color="white", lw=0.5, alpha=0.3, ls="--")
        ax1.set_ylim(0.4, 1.05)
        ax1.set_xticks(x); ax1.set_xticklabels(layers, rotation=45, fontsize=6)
        ax1.legend(fontsize=5.5, labelcolor="white",
                   facecolor="#2a2a2a", edgecolor="none", ncol=2)
        style(ax1, f"{model} — AUC (harmful vs benign probe)")

        # (b) Cosine with DIM — 3 lines (exclude DIM itself)
        ax2 = fig.add_subplot(gs[i, 1])
        for dname, col in DIR_COLORS.items():
            if dname == "DIM":
                continue
            cos_vals = [r["metrics"][dname]["cos_with_dim"] for r in res]
            ax2.plot(layers, cos_vals, color=col, marker="o", ms=4,
                     lw=1.5, label=DIR_LABELS[dname])
        ax2.axhline(0,   color="white", lw=0.5, alpha=0.3)
        ax2.axhline(0.5, color="#ffeb3b", lw=1, ls="--", alpha=0.5,
                    label="cos=0.5")
        ax2.set_ylim(-0.6, 1.05)
        ax2.legend(fontsize=5.5, labelcolor="white",
                   facecolor="#2a2a2a", edgecolor="none")
        style(ax2, f"{model} — Cosine with DIM direction")

        # (c) Fisher score comparison
        ax3 = fig.add_subplot(gs[i, 2])
        for j, (dname, col) in enumerate(DIR_COLORS.items()):
            fish = [r["metrics"][dname]["fisher"] for r in res]
            ax3.bar(x + (j-1.5)*w, fish, w, label=DIR_LABELS[dname],
                    color=col, alpha=0.85)
        ax3.set_xticks(x); ax3.set_xticklabels(layers, rotation=45, fontsize=6)
        ax3.legend(fontsize=5.5, labelcolor="white",
                   facecolor="#2a2a2a", edgecolor="none", ncol=2)
        style(ax3, f"{model} — Fisher LDA Score")

        # (d) SC-RFM: components used + alignment info
        ax4 = fig.add_subplot(gs[i, 3])
        comps     = [r["sc_info"]["components_used"]    for r in res]
        max_align = [r["sc_info"]["max_alignment"]       for r in res]
        cos_sc    = [r["sc_info"]["cos_sc_with_dim"]    for r in res]

        ax4_twin = ax4.twinx()
        ax4.bar(layers, comps, color="#b39ddb", alpha=0.6,
                width=0.6, label="components used")
        ax4_twin.plot(layers, cos_sc, color="#66bb6a", marker="^",
                      ms=4, lw=1.5, label="cos(SC-RFM, DIM)")
        ax4_twin.plot(layers, max_align, color="#ffd54f", marker="s",
                      ms=3, lw=1, ls="--", alpha=0.7, label="max alignment")
        ax4_twin.axhline(0, color="white", lw=0.5, alpha=0.3)
        ax4_twin.set_ylim(-0.5, 1.1)
        ax4.set_xlabel("Layer", color="white", fontsize=8)
        ax4.tick_params(axis="y", colors="#b39ddb", labelsize=7)
        ax4.tick_params(axis="x", colors="white", labelsize=7)
        ax4_twin.tick_params(colors="white", labelsize=7)
        ax4.set_facecolor("#1a1a1a")
        ax4.set_title(f"{model} — SC-RFM Components & Alignment",
                      color="white", fontsize=8.5, pad=5)
        for sp in ax4.spines.values():     sp.set_color("#444")
        for sp in ax4_twin.spines.values(): sp.set_color("#444")

        lines1, labs1 = ax4.get_legend_handles_labels()
        lines2, labs2 = ax4_twin.get_legend_handles_labels()
        ax4.legend(lines1 + lines2, labs1 + labs2, fontsize=5.5,
                   labelcolor="white", facecolor="#2a2a2a", edgecolor="none")

    fig.suptitle("SC-RFM vs DIM vs RFM — Full Experiment Report",
                 color="white", fontsize=13, y=1.005, fontweight="bold")

    out = OUTPUT_DIR / "sc_rfm_report.png"
    fig.savefig(out, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    logger.info("Figure → %s", out)


# ═════════════════════════════════════════════════════════════════════════════
# CONSOLE REPORT
# ═════════════════════════════════════════════════════════════════════════════

def print_report(all_results: dict):
    SEP = "─" * 115
    print(f"\n{SEP}")
    print("  SC-RFM EXPERIMENT SUMMARY")
    print(SEP)

    for model, res in all_results.items():
        lr = res.get("layer_results", [])
        if not lr:
            print(f"\n  [{model}] — no results")
            continue

        print(f"\n  ┌─ {model} {'─'*(60-len(model))}┐")

        # Per-layer table
        hdr = (f"  │ {'Layer':>5} │ {'AUC_DIM':>8} │ {'AUC_RFM_fix':>11} │ "
               f"{'AUC_RFM_ada':>11} │ {'AUC_SC':>8} │ "
               f"{'cos_RFM_fix':>11} │ {'cos_RFM_ada':>11} │ "
               f"{'cos_SC':>8} │ {'comps':>5} │")
        print(hdr)
        print("  │" + "─"*(len(hdr)-4) + "│")

        for r in lr:
            m   = r["metrics"]
            sc  = r["sc_info"]
            fb  = "⚠" if sc["fallback_used"] else " "
            print(
                f"  │ {r['layer']:>5} │ "
                f"{m['DIM']['auc']:>8.3f} │ "
                f"{m['RFM_top1_fixed']['auc']:>11.3f} │ "
                f"{m['RFM_top1_adaptive']['auc']:>11.3f} │ "
                f"{m['SC_RFM']['auc']:>8.3f} │ "
                f"{m['RFM_top1_fixed']['cos_with_dim']:>+11.3f} │ "
                f"{m['RFM_top1_adaptive']['cos_with_dim']:>+11.3f} │ "
                f"{m['SC_RFM']['cos_with_dim']:>+8.3f} │ "
                f"{sc['components_used']:>4}{fb} │"
            )

        # Summary averages
        def avg(key_path):
            keys = key_path.split(".")
            vals = []
            for r in lr:
                d = r
                for k in keys:
                    d = d[k]
                vals.append(float(d))
            return np.mean(vals)

        print("  │" + "─"*(len(hdr)-4) + "│")
        print(
            f"  │ {'MEAN':>5} │ "
            f"{avg('metrics.DIM.auc'):>8.3f} │ "
            f"{avg('metrics.RFM_top1_fixed.auc'):>11.3f} │ "
            f"{avg('metrics.RFM_top1_adaptive.auc'):>11.3f} │ "
            f"{avg('metrics.SC_RFM.auc'):>8.3f} │ "
            f"{avg('metrics.RFM_top1_fixed.cos_with_dim'):>+11.3f} │ "
            f"{avg('metrics.RFM_top1_adaptive.cos_with_dim'):>+11.3f} │ "
            f"{avg('metrics.SC_RFM.cos_with_dim'):>+8.3f} │ "
            f"{'':>5} │"
        )
        print(f"  └{'─'*(len(hdr)-4)}┘")

        # Verdict
        mean_cos_fix = avg("metrics.RFM_top1_fixed.cos_with_dim")
        mean_cos_ada = avg("metrics.RFM_top1_adaptive.cos_with_dim")
        mean_cos_sc  = avg("metrics.SC_RFM.cos_with_dim")
        mean_auc_sc  = avg("metrics.SC_RFM.auc")
        mean_auc_dim = avg("metrics.DIM.auc")

        print(f"\n  VERDICT [{model}]:")
        print(f"    Cosine alignment:  RFM_fix={mean_cos_fix:+.3f} | "
              f"RFM_ada={mean_cos_ada:+.3f} | SC_RFM={mean_cos_sc:+.3f}")

        if mean_cos_sc > 0.4:
            status = "✅ SC-RFM successfully aligns with DIM"
        elif mean_cos_sc > 0.1:
            status = "⚠️  SC-RFM partially aligns with DIM"
        else:
            status = "❌ SC-RFM still not well-aligned — check AGOP quality"

        print(f"    Status: {status}")
        print(f"    AUC trade-off: SC_RFM={mean_auc_sc:.3f} vs DIM={mean_auc_dim:.3f} "
              f"(diff={mean_auc_sc-mean_auc_dim:+.3f})")

        fallback_count = sum(1 for r in lr if r["sc_info"]["fallback_used"])
        if fallback_count > 0:
            print(f"    ⚠️  Fallback used in {fallback_count}/{len(lr)} layers "
                  f"— AGOP may not have steering-aligned components")

    print(f"\n{SEP}\n")
    print(f"  Saved pkl files:")
    for model in all_results:
        p = OUTPUT_DIR / f"{model}_SCRFM_refusal.pkl"
        if p.exists():
            print(f"    {p}")
    print(f"  Figure: {OUTPUT_DIR / 'sc_rfm_report.png'}")
    print(f"\n{SEP}\n")


# ═════════════════════════════════════════════════════════════════════════════
# CSV SAVE
# ═════════════════════════════════════════════════════════════════════════════

def save_csv(all_results: dict):
    import csv

    # Per-layer CSV
    p1 = OUTPUT_DIR / "sc_rfm_experiment.csv"
    cols = [
        "model","layer",
        "auc_dim","auc_rfm_fixed","auc_rfm_adaptive","auc_sc_rfm",
        "fisher_dim","fisher_rfm_fixed","fisher_rfm_adaptive","fisher_sc_rfm",
        "cos_rfm_fixed","cos_rfm_adaptive","cos_sc_rfm",
        "sc_components","sc_fallback","sc_max_align","mean_norm",
    ]
    with open(p1, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for model, res in all_results.items():
            for r in res.get("layer_results", []):
                m = r["metrics"]; sc = r["sc_info"]
                w.writerow({
                    "model": model, "layer": r["layer"],
                    "auc_dim":          round(m["DIM"]["auc"], 4),
                    "auc_rfm_fixed":    round(m["RFM_top1_fixed"]["auc"], 4),
                    "auc_rfm_adaptive": round(m["RFM_top1_adaptive"]["auc"], 4),
                    "auc_sc_rfm":       round(m["SC_RFM"]["auc"], 4),
                    "fisher_dim":       round(m["DIM"]["fisher"], 4),
                    "fisher_rfm_fixed": round(m["RFM_top1_fixed"]["fisher"], 4),
                    "fisher_rfm_adaptive": round(m["RFM_top1_adaptive"]["fisher"], 4),
                    "fisher_sc_rfm":    round(m["SC_RFM"]["fisher"], 4),
                    "cos_rfm_fixed":    round(m["RFM_top1_fixed"]["cos_with_dim"], 4),
                    "cos_rfm_adaptive": round(m["RFM_top1_adaptive"]["cos_with_dim"], 4),
                    "cos_sc_rfm":       round(m["SC_RFM"]["cos_with_dim"], 4),
                    "sc_components":    sc["components_used"],
                    "sc_fallback":      sc["fallback_used"],
                    "sc_max_align":     round(sc["max_alignment"], 4),
                    "mean_norm":        round(r["norm_info"]["mean_norm_pos"], 2),
                })
    logger.info("CSV → %s", p1)


In [2]:
np.random.seed(SEED)
torch.manual_seed(SEED)

logger.info("Output dir : %s", OUTPUT_DIR.resolve())
logger.info("Device     : %s", DEVICE)
logger.info("TOP_K      : %d eigenvectors for SC-RFM", TOP_K)
logger.info("RFM_ITERS  : %d", RFM_ITERS)

all_results = {}

for model_name in ["llama3.1", "qwen2.5", "gemma2"]:
    missing = []
    for label, path in [
        ("embed_dir", EMBEDDING_DIRS[model_name]),
        ("DIM pkl",   DIM_PKL_PATHS[model_name]),
        ("RFM pkl",   RFM_PKL_PATHS[model_name]),
    ]:
        if not Path(path).exists():
            missing.append(f"{label}: {path}")

    if missing:
        logger.warning("Skipping %s — missing:\n    %s",
                       model_name, "\n    ".join(missing))
        all_results[model_name] = {"layer_results": []}
        continue

    try:
        all_results[model_name] = run_model(model_name)
    except Exception as e:
        logger.error("Model %s failed: %s", model_name, e, exc_info=True)
        all_results[model_name] = {"layer_results": []}

save_csv(all_results)
plot_results(all_results)
print_report(all_results)

logger.info("Done. All outputs in: %s", OUTPUT_DIR.resolve())

05:21:39 [INFO] Output dir : /home/workspace/mad_workspace/llm_workspace/AGOPNullSpace/sc_rfm_results
05:21:39 [INFO] Device     : cuda
05:21:39 [INFO] TOP_K      : 20 eigenvectors for SC-RFM
05:21:39 [INFO] RFM_ITERS  : 3
05:21:39 [INFO] ======================================================================
05:21:39 [INFO] MODEL: llama3.1
05:21:39 [INFO] ======================================================================
05:21:46 [INFO] Layers_total=32  D=4096 | H_pos=(2000, 32, 4096)  H_neg=(14000, 32, 4096)
05:21:46 [INFO]   Layer 8: computing AGOP (adaptive BW)...
05:21:46 [INFO] Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
05:21:46 [INFO] NumExpr defaulting to 16 threads.
2026-05-20 05:21:46.767493101 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
05:21:48 [INFO]   Adaptiv

Fitting RFM with ntrain: 3200, d: 4096, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.3852682113647461 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.16886067390441895 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.2988905906677246 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 4096, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.09355664253234863 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.28385305404663086 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.30164408683776855 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 4096, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.09238100051879883 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.2905921936035156 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.3020625114440918 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 4096, and nval: 

05:21:58 [INFO]   Best RFM AUC=0.9984


Optimal M batch size: 3200


05:22:22 [INFO]   L 8 | AUC: DIM=0.674 RFM_fix=0.991 RFM_ada=0.988 SC=0.988 | cos_dim: RFM_fix=-0.068 RFM_ada=-0.098 SC=+0.132 | comps=10 fallback=False
05:22:22 [INFO]   Layer 9: computing AGOP (adaptive BW)...
05:22:23 [INFO]   Adaptive BW: base=2.6 → [1.3, 2.6, 5.2, 13.1]


...

Fitting RFM with ntrain: 3200, d: 3584, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.07866525650024414 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.28235387802124023 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.2703738212585449 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 3584, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.08813190460205078 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.2794351577758789 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.2712414264678955 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 3584, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.08830499649047852 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.2702317237854004 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.27191734313964844 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 3584, and nval: 

05:41:48 [INFO]   Best RFM AUC=0.9981


Optimal M batch size: 3200


05:42:09 [INFO]   L22 | AUC: DIM=0.765 RFM_fix=0.988 RFM_ada=0.992 SC=0.992 | cos_dim: RFM_fix=-0.350 RFM_ada=-0.299 SC=+0.302 | comps=10 fallback=False
05:42:09 [INFO] SC-RFM pkl → sc_rfm_results/gemma2_SCRFM_refusal.pkl
05:42:10 [INFO] CSV → sc_rfm_results/sc_rfm_experiment.csv
05:42:11 [INFO] Figure → sc_rfm_results/sc_rfm_report.png
05:42:11 [INFO] Done. All outputs in: /home/workspace/mad_workspace/llm_workspace/AGOPNullSpace/sc_rfm_results



───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  SC-RFM EXPERIMENT SUMMARY
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  ┌─ llama3.1 ────────────────────────────────────────────────────┐
  │ Layer │  AUC_DIM │ AUC_RFM_fix │ AUC_RFM_ada │   AUC_SC │ cos_RFM_fix │ cos_RFM_ada │   cos_SC │ comps │
  │────────────────────────────────────────────────────────────────────────────────────────────────────────│
  │     8 │    0.674 │       0.991 │       0.988 │    0.988 │      -0.068 │      -0.098 │   +0.132 │   10  │
  │     9 │    0.721 │       0.985 │       0.989 │    0.598 │      -0.067 │      -0.049 │   +0.137 │   12  │
  │    10 │    0.727 │       0.997 │       0.947 │    0.953 │      -0.067 │      -0.068 │   +0.255 │   12  │
  │    11 │    0.781 │       0.979 │       0.992 │    0.984 │      -0.321 │      -0.142 │   +0.218 │   15  │
  │    12 │    0

05:01:38 [INFO] Output dir : /home/workspace/mad_workspace/llm_workspace/AGOPNullSpace/sc_rfm_results
05:01:38 [INFO] Device     : cuda
05:01:38 [INFO] TOP_K      : 20 eigenvectors for SC-RFM
05:01:38 [INFO] RFM_ITERS  : 3
05:01:38 [INFO] ======================================================================
05:01:38 [INFO] MODEL: llama3.1
05:01:38 [INFO] ======================================================================
05:01:44 [INFO] Layers_total=32  D=4096 | H_pos=(2000, 32, 4096)  H_neg=(14000, 32, 4096)
05:01:44 [INFO]   Layer 8: computing AGOP (adaptive BW)...
05:01:44 [WARNING] xrfm not installed → linear AGOP fallback
05:02:15 [INFO]   L 8 | AUC: DIM=0.674 RFM_fix=0.991 RFM_ada=0.997 SC=0.997 | cos_dim: RFM_fix=-0.068 RFM_ada=-0.063 SC=+0.063 | comps=10 fallback=False
05:02:15 [INFO]   Layer 9: computing AGOP (adaptive BW)...
05:02:15 [WARNING] xrfm not installed → linear AGOP fallback
05:02:48 [INFO]   L 9 | AUC: DIM=0.714 RFM_fix=0.985 RFM_ada=0.999 SC=0.999 | cos_dim: R


───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  SC-RFM EXPERIMENT SUMMARY
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  ┌─ llama3.1 ────────────────────────────────────────────────────┐
  │ Layer │  AUC_DIM │ AUC_RFM_fix │ AUC_RFM_ada │   AUC_SC │ cos_RFM_fix │ cos_RFM_ada │   cos_SC │ comps │
  │────────────────────────────────────────────────────────────────────────────────────────────────────────│
  │     8 │    0.674 │       0.991 │       0.997 │    0.997 │      -0.068 │      -0.063 │   +0.063 │   10  │
  │     9 │    0.714 │       0.985 │       0.999 │    0.999 │      -0.067 │      -0.049 │   +0.049 │   11  │
  │    10 │    0.720 │       0.996 │       0.999 │    0.999 │      -0.067 │      -0.059 │   +0.059 │   13  │
  │    11 │    0.771 │       0.976 │       0.999 │    0.999 │      -0.321 │      -0.039 │   +0.039 │   10  │
  │    12 │    0

In [3]:
#!/usr/bin/env python
# coding: utf-8

"""
cr_steer_experiment.py
================================================================================
Full implementation + experiment cho Closed-form Resolvent Steering (CR-Steer)
nhằm khắc phục nhược điểm của SC-RFM (bị sụp đổ không gian Eigen ở bản Linear).

THUẬT TOÁN ĐỐI CHỨNG:
  1. DIM Direction (Baseline)
  2. RFM Top-1 (Fixed Bandwidth)
  3. RFM Top-1 (Adaptive Bandwidth)
  4. SC-RFM (Heuristic Eigenvector + ReLU)
  5. CR-Steer (Ours - Closed-form Resolvent: c* = (alpha*I - M)^-1 * r)

CHẠY 1 SHOT - KHÔNG CẦN ARGS - TỰ ĐỘNG CHẨN ĐOÁN VÀ ĐỒ THỊ HÓA
================================================================================
"""

from huggingface_hub import login
# Giữ nguyên token đăng nhập của bạn
login(token="***REMOVED***")

import os
import gc
import pickle
import logging
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from copy import deepcopy
from pathlib import Path

os.environ["CUDA_DEVICE_ORDER"]      = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch._dynamo
torch._dynamo.config.disable = True

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# ═════════════════════════════════════════════════════════════════════════════
# CONFIG (Giữ nguyên cấu trúc thư mục của bạn để không bị lỗi Paths)
# ═════════════════════════════════════════════════════════════════════════════

PROJECT_DIR = Path("/home/workspace/mad_workspace/llm_workspace/AGOPNullSpace")

EMBEDDING_DIRS = {
    "llama3.1": PROJECT_DIR / "data/embeddings/llama3.1",
    "qwen2.5":  PROJECT_DIR / "data/embeddings/qwen2.5",
    "gemma2":   PROJECT_DIR / "data/embeddings/gemma2",
}

DIM_PKL_PATHS = {
    "llama3.1": PROJECT_DIR / "data/refusal_vectors/RV/llama3.1_RV_refusal.pkl",
    "qwen2.5":  PROJECT_DIR / "data/refusal_vectors/RV/qwen2.5_RV_refusal.pkl",
    "gemma2":   PROJECT_DIR / "data/refusal_vectors/RV/gemma2_RV_refusal.pkl",
}

RFM_PKL_PATHS = {
    "llama3.1": PROJECT_DIR / "data/refusal_vectors/RFM/llama3.1_RFM_refusal.pkl",
    "qwen2.5":  PROJECT_DIR / "data/refusal_vectors/RFM/qwen2.5_RFM_refusal.pkl",
    "gemma2":   PROJECT_DIR / "data/refusal_vectors/RFM/gemma2_RFM_refusal.pkl",
}

STEERING_LAYERS = {
    "llama3.1": [8, 9, 10, 11, 12, 13, 14, 16, 18, 19],
    "qwen2.5":  [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19],
    "gemma2":   [6, 8, 10, 11, 12, 13, 14, 15, 16, 18, 22],
}

OUTPUT_DIR = Path("./cr_steer_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEED      = 42
SAMPLE_N  = 500    
TOP_K     = 20     # Cho thuật toán đối chứng SC-RFM
RFM_ITERS = 3

# ═════════════════════════════════════════════════════════════════════════════
# DATA LOADING & UTILITIES
# ═════════════════════════════════════════════════════════════════════════════

def load_pkl(path) -> np.ndarray:
    with open(path, "rb") as f:
        return pickle.load(f)

def load_embeddings(embed_dir: Path):
    def _load(fname):
        p = embed_dir / fname
        return torch.load(p, map_location="cpu").float() if p.exists() else None

    H_harmful   = _load("embeds_harmful_train_1000.pt")
    H_jailbreak = _load("embeds_jailbreak_train.pt")
    H_benign    = _load("embeds_benign_train.pt")
    H_coco_orig = _load("embeds_coconot_original.pt")
    H_coco_pref = _load("embeds_coconot_pref.pt")

    if H_harmful is None:
        raise FileNotFoundError(f"embeds_harmful_train_1000.pt not in {embed_dir}")

    if H_jailbreak is not None:
        idx = torch.randperm(H_jailbreak.size(0))[:1000]
        H_pos = torch.cat([H_harmful, H_jailbreak[idx]], dim=0)
    else:
        H_pos = H_harmful

    parts = []
    if H_benign is not None:
        parts.append(H_benign)
    if H_coco_orig is not None and H_coco_pref is not None:
        n_want = 4000 - H_coco_pref.size(0)
        idx_b  = torch.randperm(H_coco_orig.size(0))[:n_want]
        parts.append(H_coco_orig[idx_b])
        parts.append(H_coco_pref)
    elif H_coco_orig is not None:
        parts.append(H_coco_orig)
    if not parts:
        raise FileNotFoundError(f"No benign embeddings in {embed_dir}")
    H_neg = torch.cat(parts, dim=0)

    return H_pos, H_neg

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 1e-10 and nb > 1e-10 else 0.0

def fisher_score(h_pos: np.ndarray, h_neg: np.ndarray, direction: np.ndarray) -> float:
    p_pos = h_pos @ direction
    p_neg = h_neg @ direction
    mu_diff = (p_pos.mean() - p_neg.mean()) ** 2
    var_sum  = p_pos.var() + p_neg.var() + 1e-10
    return float(mu_diff / var_sum)

def auc_score(h_pos: np.ndarray, h_neg: np.ndarray, direction: np.ndarray) -> float:
    from sklearn.metrics import roc_auc_score
    scores = np.concatenate([h_pos @ direction, h_neg @ direction])
    labels = np.concatenate([np.ones(len(h_pos)), np.zeros(len(h_neg))])
    try:
        auc = roc_auc_score(labels, scores)
        return max(auc, 1.0 - auc)
    except Exception:
        return 0.5

def median_bandwidth(X: np.ndarray, n_sample: int = 400) -> float:
    idx   = np.random.choice(len(X), min(n_sample, len(X)), replace=False)
    X_sub = X[idx]
    dists = np.sqrt(((X_sub[:, None] - X_sub[None]) ** 2).sum(-1))
    upper = dists[np.triu_indices(len(X_sub), k=1)]
    return float(np.median(upper) / (2 ** 0.5)) if len(upper) else 1.0

def safe_norm(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    return v / n if n > 1e-10 else v

def standardize_gpu(X: torch.Tensor):
    mean_ = X.mean(dim=0)
    std_  = X.std(dim=0).clamp(min=1e-8)
    return (X - mean_) / std_, mean_, std_

def train_logistic_gpu(X, y, C=1.0, max_iter=500):
    w = torch.zeros(X.shape[1], dtype=torch.float32, device=X.device, requires_grad=True)
    opt = torch.optim.LBFGS(
        [w], lr=1.0, max_iter=max_iter,
        tolerance_grad=1e-6, tolerance_change=1e-6,
        history_size=10, line_search_fn="strong_wolfe",
    )
    def closure():
        opt.zero_grad()
        logits = X @ w
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y, reduction="mean")
        loss = loss + (0.5 / C) * (w @ w)
        loss.backward()
        return loss
    opt.step(closure)
    return w.detach()

# ═════════════════════════════════════════════════════════════════════════════
# AGOP ENGINE
# ═════════════════════════════════════════════════════════════════════════════

def compute_agop_matrix_rfm(H_pos: torch.Tensor, H_neg: torch.Tensor, rfm_iters: int = 3, device: str = "cpu") -> tuple:
    try:
        from xrfm import RFM
        from sklearn.metrics import roc_auc_score
    except ImportError:
        return _compute_agop_linear_full(H_pos, H_neg, device)

    dev   = torch.device(device)
    X     = torch.cat([H_pos, H_neg], dim=0).float().to(dev)
    y     = torch.cat([torch.ones(H_pos.shape[0], 1, device=dev), torch.zeros(H_neg.shape[0], 1, device=dev)])

    pos_idx   = (y.squeeze() == 1).nonzero(as_tuple=True)[0]
    neg_idx   = (y.squeeze() == 0).nonzero(as_tuple=True)[0]
    val_idx   = torch.cat([pos_idx[:max(1, int(0.2*len(pos_idx)))], neg_idx[:max(1, int(0.2*len(neg_idx)))].to(dev)])
    train_idx = torch.cat([pos_idx[max(1, int(0.2*len(pos_idx))):], neg_idx[max(1, int(0.2*len(neg_idx))):].to(dev)])
    Xtr, ytr  = X[train_idx], y[train_idx]
    Xvl, yvl  = X[val_idx],   y[val_idx]

    bw_base    = median_bandwidth(X.cpu().numpy())
    bandwidths = [bw_base * s for s in [0.5, 1.0, 2.0, 5.0]]

    best_model, best_auc = None, -1.0
    for bw in bandwidths:
        for reg in [1e-3, 1e-2]:
            try:
                m = RFM(kernel="l2_high_dim", bandwidth=bw, device=device)
                m.fit((Xtr, ytr), (Xvl, yvl), reg=reg, iters=rfm_iters, center_grads=True, early_stop_rfm=True, get_agop_best_model=True, top_k=1)
                preds = m.predict(Xvl).cpu().numpy()
                auc   = roc_auc_score(yvl.cpu().numpy(), preds)
                if auc > best_auc:
                    best_auc, best_model = auc, deepcopy(m)
            except Exception:
                pass

    if best_model is None:
        return _compute_agop_linear_full(H_pos, H_neg, device)

    agop = best_model.agop_best_model.cpu()
    S, U = torch.lobpcg(agop, k=1)
    r_top1 = U[:, 0]
    if torch.corrcoef(torch.stack([X.cpu() @ r_top1, y.squeeze().cpu()]))[0, 1] < 0:
        r_top1 = -r_top1

    return agop.numpy().astype(np.float32), r_top1.numpy()

def _compute_agop_linear_full(H_pos, H_neg, device):
    dev = torch.device(device)
    X   = torch.cat([H_pos, H_neg], dim=0).float().to(dev)
    y   = torch.cat([torch.ones(len(H_pos), device=dev), torch.zeros(len(H_neg), device=dev)])
    X_sc, mean_, std_ = standardize_gpu(X)

    best_acc, best_w = -1.0, None
    for C in [0.1, 1.0, 10.0]:
        w_sc = train_logistic_gpu(X_sc, y, C=C)
        acc  = ((X_sc @ w_sc > 0).float() == y).float().mean().item()
        if acc > best_acc:
            best_acc, best_w = acc, w_sc.clone()

    w_orig = (best_w / std_).cpu().numpy().astype(np.float32)
    agop   = np.outer(w_orig, w_orig)
    r      = w_orig / (np.linalg.norm(w_orig) + 1e-10)
    return agop, r

# ═════════════════════════════════════════════════════════════════════════════
# TOÁN HỌC MỚI: CLOSED-FORM RESOLVENT STEERING (CR-STEER)
# ═════════════════════════════════════════════════════════════════════════════

def cr_steer(agop: np.ndarray, r_dim: np.ndarray, alpha_ratio: float = 1.2) -> dict:
    """
    Closed-form Resolvent Steering (CR-Steer).
    Giải trực tiếp bài toán tối ưu hóa lồi KKT: c* = normalize( (alpha*I - M)^-1 * r )
    """
    r = safe_norm(r_dim)
    d = agop.shape[0]

    # Ước lượng nhanh lambda_max của ma trận AGOP qua Power Iteration để tối ưu tốc độ
    v = np.random.randn(d)
    v = v / np.linalg.norm(v)
    for _ in range(8):
        v = agop @ v
        v_norm = np.linalg.norm(v)
        if v_norm < 1e-9: break
        v = v / v_norm
    lambda_max = float(v @ (agop @ v))
    if lambda_max < 1e-7:
        lambda_max = 1.0

    # Tính dịch phổ alpha để bảo toàn tính xác định dương ổn định tuyệt đối
    alpha = alpha_ratio * lambda_max

    # Giải hệ phương trình tuyến tính (alpha * I - M) * c = r
    A = alpha * np.eye(d) - agop
    try:
        c = np.linalg.solve(A, r)
    except np.linalg.LinAlgError:
        # Cơ chế bù nhiễu Tikhonov nếu hệ phương trình quá suy biến (trường hợp ma trận hạng 1 tuyến tính)
        A += 0.05 * lambda_max * np.eye(d)
        c = np.linalg.solve(A, r)

    c = safe_norm(c)
    cos_with_dim = float(np.dot(c, r))

    return {
        "c_crsteer": c,
        "cos_with_dim": cos_with_dim,
        "lambda_max": lambda_max,
        "alpha": alpha
    }

def sc_rfm(agop: np.ndarray, r_dim: np.ndarray, top_k: int = 20, fallback_threshold: float = 0.05) -> dict:
    # Giữ lại hàm SC-RFM cũ của bạn làm baseline đối chứng trực tiếp
    r = safe_norm(r_dim)
    K_actual = min(top_k, agop.shape[0])
    try:
        from scipy.linalg import eigh
        eigenvalues, eigenvectors = eigh(agop, subset_by_index=[agop.shape[0] - K_actual, agop.shape[0] - 1])
        eigenvalues, eigenvectors = eigenvalues[::-1].copy(), eigenvectors[:, ::-1].copy()
    except Exception:
        eigenvalues, eigenvectors = np.linalg.eigh(agop)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues, eigenvectors = eigenvalues[idx][:K_actual], eigenvectors[:, idx][:, :K_actual]

    eigenvalues = np.maximum(eigenvalues, 0)
    alignments = eigenvectors.T @ r
    weights = eigenvalues * np.maximum(alignments, 0)

    fallback_used = False
    if weights.sum() < fallback_threshold:
        k_best = int(np.argmax(np.abs(alignments)))
        c = eigenvectors[:, k_best] * np.sign(alignments[k_best])
        fallback_used = True
        components_used = 1
    else:
        c = eigenvectors @ weights
        components_used = int((weights > 0).sum())

    c = safe_norm(c)
    return {
        "c_scrfm": c,
        "components_used": components_used,
        "cos_with_dim": float(np.dot(c, r)),
        "fallback_used": fallback_used,
        "alignments": alignments
    }

# ═════════════════════════════════════════════════════════════════════════════
# EXPERIMENT PIPELINE
# ═════════════════════════════════════════════════════════════════════════════

def run_layer(layer: int, H_pos_full: torch.Tensor, H_neg_full: torch.Tensor, r_dim_all: np.ndarray, c_rfm_all: np.ndarray, rfm_iters: int, device: str, top_k: int, sample_n: int) -> dict:
    h_pos_l = H_pos_full[:, layer, :].numpy()
    h_neg_l = H_neg_full[:, layer, :].numpy()

    n = min(len(h_pos_l), len(h_neg_l), sample_n)
    hp = h_pos_l[np.random.choice(len(h_pos_l), n, replace=False)]
    hn = h_neg_l[np.random.choice(len(h_neg_l), n, replace=False)]

    r_dim = safe_norm(r_dim_all[layer].astype(np.float32))
    c_rfm = safe_norm(c_rfm_all[layer].astype(np.float32))

    h_pos_t, h_neg_t = torch.tensor(h_pos_l).float(), torch.tensor(h_neg_l).float()
    n_min = min(len(h_pos_t), len(h_neg_t))
    if len(h_pos_t) > n_min: h_pos_t = h_pos_t[torch.randperm(len(h_pos_t))[:n_min]]
    if len(h_neg_t) > n_min: h_neg_t = h_neg_t[torch.randperm(len(h_neg_t))[:n_min]]

    logger.info("  Layer %d: computing AGOP matrix...", layer)
    agop, c_rfm_adaptive_top1 = compute_agop_matrix_rfm(h_pos_t, h_neg_t, rfm_iters=rfm_iters, device=device)

    # 1. SC-RFM (Heuristic cũ)
    sc_result = sc_rfm(agop, r_dim, top_k=top_k)
    
    # 2. CR-Steer (Toán học mới của chúng ta)
    cr_result = cr_steer(agop, r_dim, alpha_ratio=1.2)

    directions = {
        "DIM":               r_dim,
        "RFM_top1_fixed":    c_rfm,
        "RFM_top1_adaptive": c_rfm_adaptive_top1,
        "SC_RFM":            sc_result["c_scrfm"],
        "CR_Steer":          cr_result["c_crsteer"],
    }

    metrics = {}
    for name, d_vec in directions.items():
        metrics[name] = {
            "auc":          auc_score(hp, hn, d_vec),
            "fisher":       fisher_score(hp, hn, d_vec),
            "cos_with_dim": cosine_sim(d_vec, r_dim),
        }

    return {
        "layer":       layer,
        "metrics":     metrics,
        "sc_info":     sc_result,
        "cr_info":     cr_result,
        "c_scrfm":     sc_result["c_scrfm"],
        "c_crsteer":   cr_result["c_crsteer"]
    }

def run_model(model_name: str) -> dict:
    logger.info("=" * 80)
    logger.info("PROCESSING SYSTEM: %s", model_name.upper())
    logger.info("=" * 80)

    r_dim_all = load_pkl(DIM_PKL_PATHS[model_name]).astype(np.float32)
    c_rfm_all = load_pkl(RFM_PKL_PATHS[model_name]).astype(np.float32)
    H_pos_full, H_neg_full = load_embeddings(EMBEDDING_DIRS[model_name])

    L, D = H_pos_full.shape[1], H_pos_full.shape[2]
    layers = STEERING_LAYERS[model_name]
    layer_results = []
    
    scrfm_vectors = np.zeros((L, D), dtype=np.float32)
    crsteer_vectors = np.zeros((L, D), dtype=np.float32)

    for layer in layers:
        try:
            res = run_layer(layer, H_pos_full, H_neg_full, r_dim_all, c_rfm_all, rfm_iters=RFM_ITERS, device=DEVICE, top_k=TOP_K, sample_n=SAMPLE_N)
            layer_results.append(res)
            scrfm_vectors[layer] = res["c_scrfm"]
            crsteer_vectors[layer] = res["c_crsteer"]

            m = res["metrics"]
            logger.info(
                "  L%2d | AUC: DIM=%.3f SC=%.3f CR=%.3f | cos_dim: RFM_ada=%+.3f SC=%+.3f CR=%+.3f",
                layer, m["DIM"]["auc"], m["SC_RFM"]["auc"], m["CR_Steer"]["auc"],
                m["RFM_top1_adaptive"]["cos_with_dim"], m["SC_RFM"]["cos_with_dim"], m["CR_Steer"]["cos_with_dim"]
            )
        except Exception as e:
            logger.error("  Layer %d failed: %s", layer, e, exc_info=True)

    # Xuất file lưu trữ pkl cho cả hai phương pháp để bạn nạp trực tiếp vào pipeline sinh văn bản (Text Gen)
    with open(OUTPUT_DIR / f"{model_name}_SCRFM_refusal.pkl", "wb") as f:
        pickle.dump(scrfm_vectors, f)
    with open(OUTPUT_DIR / f"{model_name}_CRSTEER_refusal.pkl", "wb") as f:
        pickle.dump(crsteer_vectors, f)

    del H_pos_full, H_neg_full; gc.collect()
    return {"model": model_name, "layer_results": layer_results}

# ═════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC REPORT & VISUALIZATION
# ═════════════════════════════════════════════════════════════════════════════

def plot_results(all_results: dict):
    models = [m for m in all_results if all_results[m].get("layer_results")]
    if not models: return

    n = len(models)
    fig = plt.figure(figsize=(24, 6 * n))
    fig.patch.set_facecolor("#0b0c10")
    gs = gridspec.GridSpec(n, 3, figure=fig, hspace=0.5, wspace=0.3)

    DIR_COLORS = {
        "DIM":              "#78909c",
        "RFM_top1_fixed":   "#ef5350",
        "RFM_top1_adaptive":"#ffa726",
        "SC_RFM":           "#66bb6a",
        "CR_Steer":         "#26c6da"  # Cyan đại diện cho toán học mới lồi lanh lợi
    }

    for i, model in enumerate(models):
        res = all_results[model]["layer_results"]
        layers = [r["layer"] for r in res]
        x = np.arange(len(layers))
        w = 0.15

        # Đồ thị so sánh AUC
        ax1 = fig.add_subplot(gs[i, 0])
        for j, (dname, col) in enumerate(DIR_COLORS.items()):
            aucs = [r["metrics"][dname]["auc"] for r in res]
            ax1.bar(x + (j-2)*w, aucs, w, label=dname, color=col, alpha=0.85)
        ax1.set_facecolor("#1f2833")
        ax1.set_title(f"{model.upper()} - AUC Performance Probe", color="white", fontsize=10)
        ax1.set_xticks(x); ax1.set_xticklabels(layers, color="white")
        ax1.set_ylim(0.4, 1.05)
        ax1.legend(fontsize=6, facecolor="#0b0c10", labelcolor="white")

        # Đồ thị góc xoay Cosine với DIM
        ax2 = fig.add_subplot(gs[i, 1])
        for dname, col in DIR_COLORS.items():
            if dname == "DIM": continue
            cos_vals = [r["metrics"][dname]["cos_with_dim"] for r in res]
            ax2.plot(layers, cos_vals, color=col, marker="o", lw=1.8, label=dname)
        ax2.set_facecolor("#1f2833")
        ax2.set_title(f"{model.upper()} - Cosine Alignment with DIM (r)", color="white", fontsize=10)
        ax2.axhline(0, color="white", alpha=0.3, ls="--")
        ax2.set_ylim(-0.7, 1.05)
        ax2.legend(fontsize=6, facecolor="#0b0c10", labelcolor="white")

        # Đồ thị so sánh Fisher Score (Độ phân tách tuyến tính lớp)
        ax3 = fig.add_subplot(gs[i, 2])
        for j, (dname, col) in enumerate(DIR_COLORS.items()):
            fish = [r["metrics"][dname]["fisher"] for r in res]
            ax3.bar(x + (j-2)*w, fish, w, label=dname, color=col, alpha=0.85)
        ax3.set_facecolor("#1f2833")
        ax3.set_title(f"{model.upper()} - Fisher Separation Power", color="white", fontsize=10)
        ax3.set_xticks(x); ax3.set_xticklabels(layers, color="white")
        ax3.legend(fontsize=6, facecolor="#0b0c10", labelcolor="white")

    fig.suptitle("CR-STEER DIAGNOSTIC REPORT: CLOSED-FORM OPERATOR VS HEURISTIC SC-RFM", color="white", fontsize=14, y=0.96, fontweight="bold")
    fig.savefig(OUTPUT_DIR / "cr_steer_diagnostic_report.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

def print_terminal_report(all_results: dict):
    LINE = "═" * 125
    print(f"\n{LINE}\n\tBẢNG CHẨN ĐOÁN TOÁN HỌC MỚI: CLOSED-FORM RESOLVENT STEERING\n{LINE}")
    
    for model, res_dict in all_results.items():
        lr = res_dict.get("layer_results", [])
        if not lr: continue
        print(f"\n▶ HỆ THỐNG MÔ HÌNH: {model.upper()}")
        print(f"  {'Layer':<6}│{'AUC_DIM':<9}│{'AUC_SC':<9}│{'AUC_CR':<9}│{'cos_RFM_ada':<13}│{'cos_SC':<9}│{'cos_CR':<9}│{'Trạng thái CR':<15}")
        print("  " + "─"*115)
        
        for r in lr:
            m = r["metrics"]
            status = "✅ Ổn định lồi" if m["CR_Steer"]["cos_with_dim"] > 0 else "❌ Nghịch hướng"
            print(f"  {r['layer']:<6}│"
                  f"{m['DIM']['auc']:<9.3f}│"
                  f"{m['SC_RFM']['auc']:<9.3f}│"
                  f"{m['CR_Steer']['auc']:<9.3f}│"
                  f"{m['RFM_top1_adaptive']['cos_with_dim']:<13.3f}│"
                  f"{m['SC_RFM']['cos_with_dim']:<9.3f}│"
                  f"{m['CR_Steer']['cos_with_dim']:<9.3f}│"
                  f"{status:<15}")
    print(f"\n{LINE}\nToàn bộ kết quả, đồ thị chẩn đoán trực quan đã xuất tại: {OUTPUT_DIR.resolve()}\n{LINE}\n")

def save_csv_report(all_results: dict):
    import csv
    p = OUTPUT_DIR / "cr_steer_metrics.csv"
    cols = ["model", "layer", "auc_dim", "auc_sc_rfm", "auc_cr_steer", "cos_rfm_adaptive", "cos_sc_rfm", "cos_cr_steer", "fisher_sc_rfm", "fisher_cr_steer"]
    with open(p, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for model, res in all_results.items():
            for r in res.get("layer_results", []):
                m = r["metrics"]
                w.writerow({
                    "model": model, "layer": r["layer"],
                    "auc_dim": round(m["DIM"]["auc"], 4),
                    "auc_sc_rfm": round(m["SC_RFM"]["auc"], 4),
                    "auc_cr_steer": round(m["CR_Steer"]["auc"], 4),
                    "cos_rfm_adaptive": round(m["RFM_top1_adaptive"]["cos_with_dim"], 4),
                    "cos_sc_rfm": round(m["SC_RFM"]["cos_with_dim"], 4),
                    "cos_cr_steer": round(m["CR_Steer"]["cos_with_dim"], 4),
                    "fisher_sc_rfm": round(m["SC_RFM"]["fisher"], 4),
                    "fisher_cr_steer": round(m["CR_Steer"]["fisher"], 4)
                })

# ═════════════════════════════════════════════════════════════════════════════
# MAIN KICKSTART EXECUTION
# ═════════════════════════════════════════════════════════════════════════════

np.random.seed(SEED)
torch.manual_seed(SEED)

logger.info("Khởi tạo hệ thống kiểm thử CR-Steer tại thư mục: %s", OUTPUT_DIR.resolve())
all_results = {}

for model_name in ["llama3.1", "qwen2.5", "gemma2"]:
    missing = []
    for label, path in [
        ("embed_dir", EMBEDDING_DIRS[model_name]),
        ("DIM pkl",   DIM_PKL_PATHS[model_name]),
        ("RFM pkl",   RFM_PKL_PATHS[model_name]),
    ]:
        if not Path(path).exists():
            missing.append(f"{label}: {path}")

    if missing:
        logger.warning("Bỏ qua %s do thiếu file gốc dữ liệu:\n    %s", model_name, "\n    ".join(missing))
        continue

    try:
        all_results[model_name] = run_model(model_name)
    except Exception as e:
        logger.error("Hệ thống %s lỗi nghiêm trọng: %s", model_name, e, exc_info=True)

if all_results:
    save_csv_report(all_results)
    plot_results(all_results)
    print_terminal_report(all_results)
    logger.info("Hoàn thành! Toàn bộ file pkl lái vector cấu trúc mới đã sẵn sàng cho AlphaSteer.")
else:
    logger.error("Không có mô hình nào chạy thành công. Vui lòng kiểm tra lại đường dẫn lưu trữ embeddings.")

06:02:14 [INFO] Khởi tạo hệ thống kiểm thử CR-Steer tại thư mục: /home/workspace/mad_workspace/llm_workspace/AGOPNullSpace/cr_steer_results
06:02:14 [INFO] ================================================================================
06:02:14 [INFO] PROCESSING SYSTEM: LLAMA3.1
06:02:14 [INFO] ================================================================================
06:02:33 [INFO]   Layer 8: computing AGOP matrix...


Fitting RFM with ntrain: 3200, d: 4096, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.08693957328796387 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.2960648536682129 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.291731595993042 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 4096, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.09879517555236816 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.29224181175231934 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.2997462749481201 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 4096, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.08806347846984863 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.292985200881958 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.2942187786102295 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 4096, and nval: 800

06:03:35 [INFO]   L 8 | AUC: DIM=0.674 SC=0.988 CR=0.922 | cos_dim: RFM_ada=-0.098 SC=+0.132 CR=+0.907
06:03:35 [INFO]   Layer 9: computing AGOP matrix...


Fitting RFM with ntrain: 3200, d: 4096, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.09213638305664062 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.2860410213470459 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.28992414474487305 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 4096, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.08888626098632812 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.29583096504211426 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.2960033416748047 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 4096, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.08940482139587402 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.2907741069793701 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.29080891609191895 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 4096, and nval:

06:04:35 [INFO]   L 9 | AUC: DIM=0.700 SC=0.962 CR=0.744 | cos_dim: RFM_ada=-0.034 SC=+0.046 CR=+0.986
06:04:35 [INFO]   Layer 10: computing AGOP matrix...


...

Fitting RFM with ntrain: 3200, d: 3584, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.060806989669799805 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.20048785209655762 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.20431303977966309 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 3584, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.05689716339111328 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.19936513900756836 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.2046976089477539 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 3584, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.06447935104370117 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.19593191146850586 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.20428204536437988 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 3584, and nv

06:31:36 [INFO]   L18 | AUC: DIM=0.737 SC=0.623 CR=0.941 | cos_dim: RFM_ada=-0.220 SC=+0.282 CR=+0.758
06:31:36 [INFO]   Layer 22: computing AGOP matrix...


Fitting RFM with ntrain: 3200, d: 3584, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.06132006645202637 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.1957862377166748 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.19983386993408203 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 3584, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.06406164169311523 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.20000767707824707 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.19988298416137695 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 3584, and nval: 800
Optimal M batch size: 3200
Time taken for round 0: 0.06410765647888184 seconds
Optimal M batch size: 3200
Time taken for round 1: 0.19999217987060547 seconds
Optimal M batch size: 3200
Time taken for round 2: 0.20379281044006348 seconds
Optimal M batch size: 3200
Fitting RFM with ntrain: 3200, d: 3584, and nva

06:32:30 [INFO]   L22 | AUC: DIM=0.787 SC=0.990 CR=0.959 | cos_dim: RFM_ada=-0.299 SC=+0.302 CR=+0.665
06:32:32 [INFO] Hoàn thành! Toàn bộ file pkl lái vector cấu trúc mới đã sẵn sàng cho AlphaSteer.



═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
	BẢNG CHẨN ĐOÁN TOÁN HỌC MỚI: CLOSED-FORM RESOLVENT STEERING
═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

▶ HỆ THỐNG MÔ HÌNH: LLAMA3.1
  Layer │AUC_DIM  │AUC_SC   │AUC_CR   │cos_RFM_ada  │cos_SC   │cos_CR   │Trạng thái CR  
  ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  8     │0.674    │0.988    │0.922    │-0.098       │0.132    │0.907    │✅ Ổn định lồi  
  9     │0.700    │0.962    │0.744    │-0.034       │0.046    │0.986    │✅ Ổn định lồi  
  10    │0.710    │0.851    │0.860    │-0.059       │0.064    │0.960    │✅ Ổn định lồi  
  11    │0.787    │0.985    │0.958    │-0.145       │0.221    │0.842    │✅ Ổn định lồi  
  12    │0.730    │0.985    │0.912    │-0.091       │0.116    │0.918    │✅ Ổn định lồi  
  13